In [ ]:
%%writefile "DS project/README.md"

# End-to-End Crossword Solver System

An End-to-End German Crossword Solver using Computer Vision and NLP



## Project Overview

This project presents a fully automated end-to-end system for solving German crossword puzzles starting from a single input image.

The system takes a raw scanned or photographed crossword puzzle and processes it through a complete pipeline that includes:

- Image correction and enhancement
- Crossword grid detection and rectification
- Cell extraction and classification
- Detection of question cells and arrow cells
- Linking arrows to their corresponding questions
- Determining answer direction and length
- OCR extraction of question text
- Candidate answer generation
- Final crossword solving using AI techniques

The project integrates multiple AI disciplines into one coherent system.



## Core Technologies and Concepts

The system combines:

- Computer Vision
- Image Processing
- Optical Character Recognition (OCR)
- Large Language Models (LLMs)
- Web Scraping
- Constraint Satisfaction Problems (CSP)



## Key Features

- Fully automated pipeline from image to solved crossword
- Advanced deskewing and perspective correction
- Accurate crossword grid detection
- Robust grid-to-cell decomposition
- Detection of question and arrow cells
- Correct arrow-to-question association
- Automatic answer direction detection (Right / Down)
- Automatic answer length computation
- OCR of German crossword clues using OpenAI Vision
- Candidate answer generation via:
  - Web scraping (wort-suchen.de)
  - Large Language Models (LLMs)
- Hybrid crossword solving using:
  - Letter propagation
  - CSP backtracking
- Structured CSV outputs for evaluation and reuse



## Technologies Used

### Programming Language
- Python 3.10

### Execution Environment
- Google Colab (recommended)
- Jupyter Notebook (local alternative)

### Libraries and Tools
- OpenCV – Computer vision and image processing
- NumPy, Pandas – Data processing
- Matplotlib – Visualization
- Requests, BeautifulSoup – Web scraping
- ipywidgets – Interactive image upload
- OpenAI API – OCR and LLM reasoning



## Environment Setup and How to Run (Beginner Friendly)

### 1. Installing Python (Local Execution)

If running locally:

1. Download Python 3.10 from:
   https://www.python.org/downloads
2. During installation, enable **Add Python to PATH**
3. Verify installation:

```bash
python --version
````



### 2. Running on Google Colab (Recommended)

1. Open: [https://colab.research.google.com](https://colab.research.google.com)
2. Click **New Notebook**
3. From the menu:

   * Runtime → Change runtime type
   * Language: Python
   * Hardware accelerator: Optional (GPU)



### 3. Installing Required Libraries

Run once:

```bash
!pip install pandas requests beautifulsoup4 numpy matplotlib \
opencv-python-headless imutils ipywidgets openai
```



### 4. OpenAI API Key Setup

```python
import os
os.environ["OPENAI_API_KEY"] = "PUT_THE_API_KEY_HERE"
```



### 5. Uploading a Crossword Image

* Upload one image at a time
* Recommended:

  * Clear printed crossword
  * Newspaper or magazine scan
  * Regular grid layout



### 6. Running the Project

* Execute notebook cells in order
* Each stage depends on the previous one
* No manual intervention is required after image upload



## Project Structure

```text
DS project/
│
├── uploaded_images/          # User-uploaded images
├── results/                  # Resized images
├── results_deskew/           # Deskewed images
├── results_warp/             # Perspective-corrected grids
├── results_grid/
│   ├── grid_data.npz
│   ├── text_cells.npy
│   ├── arrow_heads.npy
│   ├── FINAL_QA_COMPLETE.csv
│   ├── OCR_questions_FINAL.csv
│   └── Final solved outputs
```



## Processing Pipeline (Stages)

### Stage 1 — Deskew

Automatic correction of image rotation for grid alignment.

### Stage 2 — Grid Warping and Rectification

Includes:

* Auto Warp
* Grid-based Warp
* Perspective Rectification
* Masked grid-only image generation

### Stage 3 — Binary Conversion and Grid Line Extraction

* Strong binarization (CamScanner-like)
* Robust grid line extraction

### Stage 4 — Perfect Uniform Grid Construction

* Builds a clean, regular grid
* Corrects row and column distortions

### Stage 5 — Cell Extraction

* Extracts all cells accurately
* Enforces a 13 × 10 grid
* Safe manual masking for special regions

### Stage 6 — Question Cell Detection

* Detects text and question cells
* Removes noise and grid residuals

#### Stage 6.1 — Center-Box Recovery

* Recovers difficult cells
* Uses ink density, spread, and component shape constraints

### Stage 7 — Arrow Detection

* Detects large and small arrow heads
* Determines arrow direction
* Saves final arrow metadata

### Stage 8 — Arrow-to-Question Association

* Links each arrow to the correct question
* Supports multi-arrow cases
* Determines:

  * Question
  * Answer direction
  * Answer start cell

### Stage 9 — Answer Length Determination

* Traverses the grid from the arrow start cell
* Stops at boundaries, masks, or other questions
* Computes answer length automatically
* Includes visual validation

### Stage 10 — Final Merged QA Result

**(Question + Answer + Direction + Length)**

* Merges all extracted information
* Outputs a final structured CSV



## OCR and Hybrid Crossword Solver

* OCR of question text using OpenAI Vision
* Candidate generation via:

  * wort-suchen.de (Web Scraping)
  * Large Language Models
* Hybrid solving strategy:

  * Letter propagation
  * CSP backtracking
* Ensures no question ends with zero candidates
* Final results stored in CSV files



## Evaluation

### Quantitative Evaluation

* Dataset of 15 crossword images
* Accuracy computed per image
* Comparison with ground truth

### Visual Evaluation

Manual inspection of:

* Grid correctness
* Cell segmentation
* Arrow–question linking
* Direction and length correctness
* Final solution plausibility

Combining quantitative and visual evaluation provides a realistic system assessment.



## Challenges and Solutions

### Challenges

* Variable image quality
* OCR errors in German text
* Arrow versus text confusion
* Intersecting answers

### Solutions

* Advanced deskewing
* Grid-based rectification
* Intelligent masking
* Hybrid CSP and LLM solver



## Future Work

* Multi-language support
* Custom-trained OCR model
* Real-time optimization
* Support for non-standard grids



## Conclusion

This project demonstrates the design of a complete end-to-end AI system capable of solving crossword puzzles automatically from a single image by integrating computer vision, natural language processing, and symbolic problem-solving techniques.



In [ ]:
# =========================================================
# Environment Setup & Library Installation
# =========================================================
# This cell installs all required third-party libraries.
# Must be executed once before running the project.
# =========================================================

!pip install -q pandas requests beautifulsoup4 numpy matplotlib \
opencv-python-headless imutils ipywidgets openai

# =========================================================
# OpenAI Client Import
# =========================================================
# Used for OCR (Vision) and LLM-based reasoning
# =========================================================

from openai import OpenAI

# =========================================================
# Standard Python Libraries
# =========================================================
# Built-in libraries for file handling, utilities, and helpers
# =========================================================

import os
import time
import json
import re
import ast
import csv
import base64
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from urllib.parse import quote

# =========================================================
# Third-Party Libraries
# =========================================================
# Core libraries for computer vision, data processing,
# visualization, and web scraping
# =========================================================

import numpy as np
import pandas as pd
import cv2
import imutils
import matplotlib.pyplot as plt
import requests
from bs4 import BeautifulSoup

# =========================================================
# Jupyter / Google Colab Utilities
# =========================================================
# Widgets and display helpers for interactive usage
# =========================================================

from IPython.display import display, clear_output
from ipywidgets import FileUpload

In [ ]:
# =====================================
# Setup
# =====================================
BASE_DIR = Path("DS project")
IMG_DIR = BASE_DIR / "uploaded_images"
IMG_DIR.mkdir(parents=True, exist_ok=True)

CURRENT_IMAGES = []

# =====================================
# Helpers
# =====================================
def clear_old_images():
    for f in IMG_DIR.glob("*"):
        try:
            f.unlink()
        except Exception:
            pass
    CURRENT_IMAGES.clear()

# =====================================
# Upload widget
# =====================================
uploader = FileUpload(accept="image/*", multiple=False)

print(" Please upload ONE image to start")
display(uploader)

# =====================================
# Upload handler (FINAL FIX)
# =====================================
def on_upload_change(change):
    clear_output(wait=True)
    print(" Clearing old images...")
    clear_old_images()

    uploaded = uploader.value

    if not uploaded:
        print(" Please upload an image")
        display(uploader)
        return

    # ---------- CASE 1: dict (Colab / new ipywidgets) ----------
    if isinstance(uploaded, dict):
        filename, fileinfo = next(iter(uploaded.items()))
        content = fileinfo["content"]

    # ---------- CASE 2: list ----------
    elif isinstance(uploaded, (list, tuple)):
        fileinfo = uploaded[0]
        filename = fileinfo["name"]
        content  = fileinfo["content"]

    else:
        raise TypeError(f"Unexpected upload format: {type(uploaded)}")

    ext = os.path.splitext(filename)[1].lower()
    save_path = IMG_DIR / f"input{ext}"

    with open(save_path, "wb") as f:
        f.write(content)

    CURRENT_IMAGES.append(save_path)

    print(f" Image saved: {save_path}")
    print(" Ready to continue pipeline")

    print("\n CURRENT_IMAGES:")
    for p in CURRENT_IMAGES:
        print("   →", p)


display(uploader)
uploader.observe(on_upload_change, names="value")


In [ ]:
# %% ===== Stufe 0: Setup & Einlesen (nur neue Bilder) =====
# This stage reads ONLY the newly uploaded images (CURRENT_IMAGES),
# resizes them if needed, and saves them into a results folder.


# === Define directories ===
BASE_DIR = Path("DS project")
IMG_DIR = BASE_DIR / "uploaded_images"     # Same folder used for upload
OUT_BASE = BASE_DIR / "results"
OUT_BASE.mkdir(exist_ok=True, parents=True)

# === Ensure the upload list exists ===
try:
    CURRENT_IMAGES
except NameError:
    raise ValueError(" CURRENT_IMAGES not found! You must upload images first.")

if len(CURRENT_IMAGES) == 0:
    raise FileNotFoundError(" No new images uploaded. Please upload images first!")

# === Image display/save helper ===
def show_and_save(title, img, name, out_dir):
    plt.figure(figsize=(9, 9))
    if img.ndim == 2:
        plt.imshow(img, cmap="gray")
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

    path = out_dir / name
    to_save = img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    cv2.imwrite(str(path), to_save)
    print("Gespeichert:", path)

# === Resize helper ===
def ensure_max_side(img, max_side=1600):
    h, w = img.shape[:2]
    if max(h, w) <= max_side:
        return img
    scale = max_side / max(h, w)
    return cv2.resize(img, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_CUBIC)

# === Process ONLY the newly uploaded images ===
print(f" {len(CURRENT_IMAGES)} neue Bilder gefunden:")

for img_path in CURRENT_IMAGES:
    img_path = Path(img_path)     # ensure Path object
    print("-", img_path.name)

print("\n Beginne Verarbeitung der neuen Bilder...\n")

for img_path in CURRENT_IMAGES:
    name = img_path.stem
    print(f"\n Verarbeite: {name}")

    # Create output folder for this image
    OUT_DIR = OUT_BASE / name
    OUT_DIR.mkdir(exist_ok=True)

    # Read image
    img0 = cv2.imread(str(img_path))
    assert img0 is not None, f" Konnte das Bild nicht lesen: {img_path}"

    # Resize
    img0 = ensure_max_side(img0, 1600)
    print("Originalgröße:", img0.shape)

    # Save preview
    show_and_save("Original", img0, "00_original.jpg", OUT_DIR)

print("\n Neue Bilder wurden verarbeitet und gespeichert unter:", OUT_BASE)


In [ ]:
# %% ===== new Stufe 1: Deskew (nur neue Bilder) – IMPROVED VERSION =====


# === Directories ===
BASE_DIR = Path("DS project")
OUT_BASE = BASE_DIR / "results_deskew"
OUT_BASE.mkdir(parents=True, exist_ok=True)

# === Ensure CURRENT_IMAGES exists ===
try:
    CURRENT_IMAGES
except NameError:
    raise ValueError("CURRENT_IMAGES not found! Please upload images first.")

if len(CURRENT_IMAGES) == 0:
    raise FileNotFoundError("No new images uploaded. Please upload images first!")


# === Display & Save helper (same signature as your notebook) ===
def show_and_save(title, img, filename, out_dir):
    plt.figure(figsize=(8, 8))
    if img.ndim == 2:
        plt.imshow(img, cmap="gray")
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()
    cv2.imwrite(str(out_dir / filename), img)
    print(" Gespeichert:", out_dir / filename)


# === Step 1: Hough-based angle detection ===
def compute_angle_hough(gray):
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blur, 50, 150)

    # strengthen grid lines
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel)

    lines = cv2.HoughLinesP(edges, 1, np.pi/180,
                            threshold=120, minLineLength=80, maxLineGap=20)
    if lines is None:
        return None

    angles = []
    for x1, y1, x2, y2 in lines[:,0]:
        angle = np.degrees(np.arctan2((y2 - y1), (x2 - x1)))
        if abs(angle) < 45:   # ignore vertical lines
            angles.append(angle)

    if len(angles) == 0:
        return None

    return float(np.median(angles))


# === Step 2: Fallback using minAreaRect ===
def compute_angle_fallback(gray):
    _, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    cnts, _ = cv2.findContours(bw, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return 0.0

    largest = max(cnts, key=cv2.contourArea)
    rect = cv2.minAreaRect(largest)
    angle = rect[-1]

    # normalize OpenCV angle to [-45,+45]
    if angle < -45:
        angle += 90

    return float(angle)


# === Step 3: Full deskew ===
def deskew_image(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    angle = compute_angle_hough(gray)
    if angle is None:
        angle = compute_angle_fallback(gray)

    rotated = imutils.rotate_bound(img, -angle)
    return rotated, angle


# === Helper: resize ===
def ensure_max_side(img, max_side=1600):
    h, w = img.shape[:2]
    if max(h, w) <= max_side:
        return img
    s = max_side / max(h, w)
    return cv2.resize(img, (int(w * s), int(h * s)), interpolation=cv2.INTER_CUBIC)


# === Process ONLY NEW images ===
all_images = [Path(p) for p in CURRENT_IMAGES]
print(f" {len(all_images)} neue Bilder werden deskewed:")

for img_path in all_images:
    name = img_path.stem
    print(f"\n Deskew – Verarbeite: {name}")

    OUT_DIR = OUT_BASE / name
    OUT_DIR.mkdir(exist_ok=True)

    img0 = cv2.imread(str(img_path))
    if img0 is None:
        print(" Konnte nicht lesen, übersprungen.")
        continue

    img0 = ensure_max_side(img0, 1600)
    img1, angle = deskew_image(img0)

    print(f" detected angle: {angle:.2f}°")

    show_and_save("Nach Deskew", img1, "01_deskew.jpg", OUT_DIR)

print("\n Fertig – neue Bilder erfolgreich deskewed!")


In [ ]:
# %% =====new Stufe 2: Auto-Warp (nur neue Bilder) – IMPROVED VERSION =====

# === Define input/output directories ===
BASE_DIR  = Path("DS project")
INPUT_DIR = BASE_DIR / "results_deskew"
OUT_BASE  = BASE_DIR / "results_warp"
OUT_BASE.mkdir(parents=True, exist_ok=True)

# === Ensure CURRENT_IMAGES exists ===
try:
    CURRENT_IMAGES
except NameError:
    raise ValueError(" CURRENT_IMAGES not found! Upload images first.")

if len(CURRENT_IMAGES) == 0:
    raise ValueError(" No new images to process! Upload images first.")


# === Helper: display and save ===
def show_and_save(title, img, filename, out_dir):
    plt.figure(figsize=(8, 8))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

    out_path = out_dir / filename
    cv2.imwrite(str(out_path), img)
    print(" Gespeichert:", out_path)


# === Improved contour-based auto-warp ===
def auto_warp(img):

    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    g = cv2.GaussianBlur(g, (5, 5), 0)

    # strong edge detection
    ed = cv2.Canny(g, 40, 160)

    # closing → connect broken lines
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    ed = cv2.morphologyEx(ed, cv2.MORPH_CLOSE, kernel, iterations=2)

    # find external contours
    cnts, _ = cv2.findContours(ed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return img

    # pick largest contour (the page)
    c = max(cnts, key=cv2.contourArea)
    peri = cv2.arcLength(c, True)
    approx = cv2.approxPolyDP(c, 0.015 * peri, True)

    # if we fail → return original
    if len(approx) != 4:
        return img

    # convert to float
    quad = approx.reshape(4, 2).astype("float32")

    # reorder points
    s = quad.sum(1)
    d = np.diff(quad, axis=1).ravel()

    rect = np.zeros((4, 2), dtype="float32")
    rect[0] = quad[np.argmin(s)]      # top-left
    rect[2] = quad[np.argmax(s)]      # bottom-right
    rect[1] = quad[np.argmin(d)]      # top-right
    rect[3] = quad[np.argmax(d)]      # bottom-left

    # output size
    W = int(max(
        np.linalg.norm(rect[2] - rect[3]),
        np.linalg.norm(rect[1] - rect[0])
    ))

    H = int(max(
        np.linalg.norm(rect[1] - rect[2]),
        np.linalg.norm(rect[0] - rect[3])
    ))

    dst = np.array([[0,0], [W-1,0], [W-1,H-1], [0,H-1]], dtype="float32")
    M   = cv2.getPerspectiveTransform(rect, dst)
    warped = cv2.warpPerspective(img, M, (W, H))

    return warped


# === Process ONLY the new uploaded images ===
for img_path in CURRENT_IMAGES:
    name = Path(img_path).stem
    print(f"\n Auto-Warp – Verarbeite: {name}")

    IN_DIR  = INPUT_DIR / name
    OUT_DIR = OUT_BASE / name
    OUT_DIR.mkdir(exist_ok=True)

    deskew_img_path = IN_DIR / "01_deskew.jpg"
    if not deskew_img_path.exists():
        print(f" Deskew file missing for {name}, skipping...")
        continue

    img = cv2.imread(str(deskew_img_path))
    if img is None:
        print(f" Cannot read {deskew_img_path}, skipping...")
        continue

    warped = auto_warp(img)
    show_and_save("Nach Auto-Warp", warped, "02_warp.jpg", OUT_DIR)

print("\n Fertig – nur neue Bilder wurden auto-warped gespeichert!")


In [ ]:
# %% ===== new Stufe 2.1: GRID-BASED WARP (REPLACES AUTO-WARP) =====


BASE_DIR  = Path("DS project")
INPUT_DIR = BASE_DIR / "results_deskew"
OUT_BASE  = BASE_DIR / "results_warp"
OUT_BASE.mkdir(parents=True, exist_ok=True)

# === Helper: show + save (same style) ===
def show_and_save(title, img, filename, out_dir):
    plt.figure(figsize=(8, 8))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

    out_path = out_dir / filename
    cv2.imwrite(str(out_path), img)
    print(" Gespeichert:", out_path)


# -----------------------------------------
# GRID MASK (early & strong)
# -----------------------------------------
def early_grid_mask(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    bw = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY_INV,
        15, 3
    )

    h, w = bw.shape
    v_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, h//20))
    h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (w//20, 1))

    vertical   = cv2.morphologyEx(bw, cv2.MORPH_OPEN, v_kernel, iterations=2)
    horizontal = cv2.morphologyEx(bw, cv2.MORPH_OPEN, h_kernel, iterations=2)

    grid = cv2.bitwise_or(vertical, horizontal)
    grid = cv2.morphologyEx(grid, cv2.MORPH_CLOSE,
                            np.ones((5,5), np.uint8), iterations=2)
    return grid


# -----------------------------------------
# GRID ANGLE (real reference)
# -----------------------------------------
def compute_grid_angle(grid_mask):
    lines = cv2.HoughLinesP(
        grid_mask, 1, np.pi/180,
        threshold=200,
        minLineLength=150,
        maxLineGap=20
    )

    if lines is None:
        return 0.0

    angles = []
    for x1,y1,x2,y2 in lines[:,0]:
        ang = np.degrees(np.arctan2(y2-y1, x2-x1))
        if abs(ang) < 30:
            angles.append(ang)

    return float(np.median(angles)) if angles else 0.0


# -----------------------------------------
# MAIN LOOP
# -----------------------------------------
for img_path in CURRENT_IMAGES:
    name = Path(img_path).stem
    print(f"\n Stufe 2 GRID – Verarbeite: {name}")

    IN_DIR  = INPUT_DIR / name
    OUT_DIR = OUT_BASE / name
    OUT_DIR.mkdir(exist_ok=True)

    deskew_path = IN_DIR / "01_deskew.jpg"
    if not deskew_path.exists():
        print(" Deskew missing, skipping.")
        continue

    img = cv2.imread(str(deskew_path))
    if img is None:
        print(" Cannot read image.")
        continue

    # 1) detect grid early
    grid_mask = early_grid_mask(img)

    # 2) deskew based on GRID (not paper)
    angle = compute_grid_angle(grid_mask)
    print(f" Grid angle: {angle:.2f}°")

    aligned = imutils.rotate_bound(img, -angle)
    grid_mask = early_grid_mask(aligned)

    # 3) crop to grid bounding box
    cnts,_ = cv2.findContours(grid_mask,
                              cv2.RETR_EXTERNAL,
                              cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        print(" Grid not found, using full image.")
        final = aligned
    else:
        c = max(cnts, key=cv2.contourArea)
        x,y,w,h = cv2.boundingRect(c)
        final = aligned[y:y+h, x:x+w]

    show_and_save("02 Grid-Aligned Warp", final, "02_warp.jpg", OUT_DIR)

print("\n Stufe 2 GRID finished — auto-warp fully replaced.")


In [ ]:
# %% ===== Stufe 2.2 PRO: GRID RECTIFICATION (Perspective Unwarp) =====


BASE_WARP = Path("DS project/results_warp")

# ---------- helpers ----------
def show_and_save(title, img, path):
    plt.figure(figsize=(8,8))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()
    cv2.imwrite(str(path), img)
    print(" Saved:", path)


def order_points(pts):
    pts = np.array(pts, dtype="float32")
    s = pts.sum(axis=1)
    diff = np.diff(pts, axis=1)

    return np.array([
        pts[np.argmin(s)],      # top-left
        pts[np.argmin(diff)],   # top-right
        pts[np.argmax(s)],      # bottom-right
        pts[np.argmax(diff)]    # bottom-left
    ], dtype="float32")


# ---------- MAIN ----------
for img_path in CURRENT_IMAGES:

    name = Path(img_path).stem
    folder = BASE_WARP / name

    print(f"\n Stufe 2.2 GRID RECTIFY – {name}")

    #  INPUT = output of Stufe 2
    warp_path = folder / "02_warp.jpg"
    if not warp_path.exists():
        print("⚠ Missing 02_warp.jpg, skipping")
        continue

    img = cv2.imread(str(warp_path))
    if img is None:
        print("⚠ Cannot read image")
        continue

    H, W = img.shape[:2]

    # --- detect grid mask (reuse your proven function) ---
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    bw = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY_INV,
        15, 3
    )

    v_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, H//20))
    h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (W//20, 1))

    vertical   = cv2.morphologyEx(bw, cv2.MORPH_OPEN, v_kernel, iterations=2)
    horizontal = cv2.morphologyEx(bw, cv2.MORPH_OPEN, h_kernel, iterations=2)

    grid_mask = cv2.bitwise_or(vertical, horizontal)
    grid_mask = cv2.morphologyEx(
        grid_mask,
        cv2.MORPH_CLOSE,
        np.ones((5,5), np.uint8),
        iterations=2
    )

    # --- find grid contour ---
    cnts,_ = cv2.findContours(
        grid_mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if not cnts:
        print("⚠ Grid contour not found")
        continue

    c = max(cnts, key=cv2.contourArea)

    peri = cv2.arcLength(c, True)
    approx = cv2.approxPolyDP(c, 0.02 * peri, True)

    if len(approx) < 4:
        print("⚠ Could not approximate grid quadrilateral")
        continue

    # use convex hull if needed
    if len(approx) > 4:
        hull = cv2.convexHull(approx)
        approx = cv2.approxPolyDP(
            hull,
            0.02 * cv2.arcLength(hull, True),
            True
        )

    if len(approx) != 4:
        print("⚠ Grid is not 4-point shape, skipping")
        continue

    src = order_points(approx.reshape(4,2))

    # --- destination rectangle ---
    dst = np.array([
        [0, 0],
        [W-1, 0],
        [W-1, H-1],
        [0, H-1]
    ], dtype="float32")

    # --- perspective transform ---
    M = cv2.getPerspectiveTransform(src, dst)
    rectified = cv2.warpPerspective(img, M, (W, H))

    out_path = folder / "02c_grid_rectified.jpg"
    show_and_save("02.2 Grid Rectified", rectified, out_path)

print("\n Stufe 2.2 GRID RECTIFICATION DONE")


In [ ]:
# %% ===== Stufe 2.3 PRO (SAFE VERSION) — Save separate masked image for GRID ONLY =====


BASE_WARP = Path("DS project/results_warp")

def hard_mask_answer_block(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Step 1: detect dark region in bottom-right
    _, th = cv2.threshold(gray, 0, 255,
                          cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    H, W = gray.shape
    roi = th[int(H*0.55):H, int(W*0.55):W]

    cnts, _ = cv2.findContours(roi, cv2.RETR_EXTERNAL,
                               cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return img

    c = max(cnts, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(c)

    pad = 25
    x1 = max(0, int(W*0.55) + x - pad)
    y1 = max(0, int(H*0.55) + y - pad)
    x2 = min(W, int(W*0.55) + x + w + pad)
    y2 = min(H, int(H*0.55) + y + h + pad)

    img2 = img.copy()

    # Step 2: fill region (for GRID only)
    cv2.rectangle(img2, (x1, y1), (x2, y2), (180, 180, 180), -1)

    # Step 3: heavy blur
    blur = cv2.GaussianBlur(img2[y1:y2, x1:x2], (35, 35), 0)
    img2[y1:y2, x1:x2] = blur

    return img2


print("\n Stufe 2.5 PRO SAFE — Non-destructive masking\n")

for img_path in CURRENT_IMAGES:

    name = Path(img_path).stem
    print(f"Processing {name}")

    warp_folder = BASE_WARP / name
    warp_img_path = warp_folder / "02c_grid_rectified.jpg"

    if not warp_img_path.exists():
        print("⚠ Missing warp, skipping.")
        continue

    # Load original auto-warp (we DO NOT overwrite it)
    img = cv2.imread(str(warp_img_path))

    # Create masked version ONLY for grid detection
    masked_for_grid = hard_mask_answer_block(img)

    # Save new file (safe)
    out_path = warp_folder / "02_warp_maskprep.jpg"
    cv2.imwrite(str(out_path), masked_for_grid)

    print(f" Saved SAFE maskprep → {out_path}")

print("\n Done — Stufe 3 will now use the maskprep version safely!")


In [ ]:
# %% ===== new Stufe 3 PRO (Ultra): Binary + Grid Lines (Strong like CamScanner) =====


BASE_WARP = Path("DS project/results_warp")

try:
    CURRENT_IMAGES
except NameError:
    raise ValueError(" CURRENT_IMAGES not defined! Upload images first.")
if len(CURRENT_IMAGES) == 0:
    raise ValueError(" No new images uploaded yet!")

# -------------------------------
# 1) White balance correction
# -------------------------------
def white_balance(img):
    result = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(result)
    clahe = cv2.createCLAHE(clipLimit=3.5, tileGridSize=(8,8))
    l = clahe.apply(l)
    result = cv2.merge([l,a,b])
    result = cv2.cvtColor(result, cv2.COLOR_LAB2BGR)
    return result

# -------------------------------
# 2) XDoG filtering (CamScanner-like)
# -------------------------------
def xdog(img_gray, gamma=0.98, sigma=0.8, k=2.0, eps=-0.1, phi=200):
    G1 = cv2.GaussianBlur(img_gray, (0,0), sigma)
    G2 = cv2.GaussianBlur(img_gray, (0,0), sigma*k)
    D = G1 - gamma*G2
    X = np.where(D >= eps, 1, 1+np.tanh(phi*(D-eps)))
    return (X*255).astype(np.uint8)

# -------------------------------
# 3) Sauvola threshold (superior)
# -------------------------------
def sauvola(gray, w=25, k=0.34):
    mean = cv2.boxFilter(gray, cv2.CV_32F, (w,w))
    sqmean = cv2.boxFilter(gray*gray, cv2.CV_32F, (w,w))
    variance = sqmean - mean*mean
    std = np.sqrt(np.maximum(variance, 0))

    R = 128
    thresh = mean*(1 + k*((std/R)-1))
    out = (gray < thresh).astype(np.uint8)*255
    return out

# -------------------------------
# 4) show + save
# -------------------------------
def show_and_save(title, img, filename, out_dir):
    plt.figure(figsize=(8, 8))
    if img.ndim == 2:
        plt.imshow(img, cmap='gray')
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title); plt.axis("off"); plt.show()

    cv2.imwrite(str(out_dir / filename), img)
    print(" Gespeichert:", out_dir / filename)

# -------------------------------
# MAIN LOOP
# -------------------------------
for img_path in CURRENT_IMAGES:

    name = Path(img_path).stem
    print(f"\n Stufe 3 PRO – Verarbeite: {name}")

    folder = BASE_WARP / name
    warp_path = folder / "02_warp_maskprep.jpg"


    if not warp_path.exists():
        print(f" Keine 02_warp.jpg in {folder}, übersprungen.")
        continue

    img = cv2.imread(str(warp_path))
    if img is None:
        print(f" Fehler beim Lesen: {warp_path}")
        continue

    # 1) white balance + denoise
    wb = white_balance(img)

    # 2) convert + sharpen
    gray = cv2.cvtColor(wb, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (3,3), 0)

    # 3) XDoG edge enhancement
    edge_map = xdog(gray, sigma=0.7, k=1.6)

    # 4) Sauvola threshold
    bw = sauvola(edge_map, w=25, k=0.36)

    show_and_save("03 Binary (Sauvola + XDoG)", bw, "03_binary.jpg", folder)

    # --------------------------------------
    # Strong line extraction (pro level)
    # --------------------------------------
    H, W = bw.shape
    approx_cell_w = max(10, W // 18)    # dynamic cell estimation
    approx_cell_h = max(10, H // 18)

    h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (approx_cell_w, 1))
    v_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, approx_cell_h))

    h_bin = cv2.morphologyEx(bw, cv2.MORPH_OPEN, h_kernel, iterations=2)
    v_bin = cv2.morphologyEx(bw, cv2.MORPH_OPEN, v_kernel, iterations=2)

    show_and_save("04 Horizontal lines", h_bin, "04_hlines.jpg", folder)
    show_and_save("05 Vertical lines", v_bin, "05_vlines.jpg", folder)

    # grid mask
    grid_mask = cv2.bitwise_or(h_bin, v_bin)
    grid_mask = cv2.morphologyEx(grid_mask, cv2.MORPH_CLOSE, np.ones((3,3), np.uint8))

    show_and_save("06 Grid mask", grid_mask, "06_gridmask.jpg", folder)

print("\n Stufe 3 PRO fertig — Ultra quality grid extraction complete!")


In [ ]:
# %% =====new  Stufe 3.1 PRO: Refine Grid Lines + Extract Perfect Grid =====



BASE_WARP = Path("DS project/results_warp")

def show(title, img):
    plt.figure(figsize=(6,6))
    if img.ndim == 2:
        plt.imshow(img, cmap='gray')
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()


# --- Helper: Merge close parallel lines ---
def merge_lines(lines, thresh=10):
    if len(lines) == 0:
        return []
    lines = sorted(lines)
    merged = [lines[0]]
    for l in lines[1:]:
        if abs(l - merged[-1]) < thresh:
            merged[-1] = (merged[-1] + l) // 2
        else:
            merged.append(l)
    return merged


# --- Hough detection helper ---
def detect_lines(bin_img, vertical=True):
    lines = cv2.HoughLinesP(
        bin_img, 1, np.pi/180, threshold=50,
        minLineLength=bin_img.shape[1]//3,
        maxLineGap=30
    )
    if lines is None:
        return []

    coords = []
    for x1, y1, x2, y2 in lines[:,0]:
        if vertical:
            if abs(x1 - x2) < 10:
                coords.append((x1 + x2)//2)
        else:
            if abs(y1 - y2) < 10:
                coords.append((y1 + y2)//2)

    return coords


# --- Intersection matrix ---
def build_intersections(v_lines, h_lines):
    pts = []
    for y in h_lines:
        for x in v_lines:
            pts.append((x, y))
    return pts


# ======================= MAIN LOOP =======================
for img_path in CURRENT_IMAGES:

    name = Path(img_path).stem
    print(f"\n Stufe 3.5 PRO – Verarbeite: {name}")

    folder = BASE_WARP / name

    # Load binaries from Stufe 3 PRO
    h_bin = cv2.imread(str(folder / "04_hlines.jpg"), 0)
    v_bin = cv2.imread(str(folder / "05_vlines.jpg"), 0)
    grid_mask = cv2.imread(str(folder / "06_gridmask.jpg"), 0)

    if h_bin is None or v_bin is None or grid_mask is None:
        print(" Missing Stufe 3 outputs, skipping...")
        continue

    # --- Detect lines via Hough ---
    print(" Detecting vertical + horizontal lines...")
    v_lines = detect_lines(v_bin, vertical=True)
    h_lines = detect_lines(h_bin, vertical=False)

    # --- Merge close lines ---
    v_lines = merge_lines(v_lines, thresh=20)
    h_lines = merge_lines(h_lines, thresh=20)

    v_lines = sorted(set(v_lines))
    h_lines = sorted(set(h_lines))

    # --- Build intersections ---
    pts = build_intersections(v_lines, h_lines)

    # ======================================================
    #  FINAL FIX: Filter points using REAL grid mask
    # ======================================================
    grid_mask_bin = cv2.threshold(grid_mask, 127, 255, cv2.THRESH_BINARY)[1]

    filtered_pts = []
    for (x, y) in pts:
        # keep only points ON white lines (part of the actual grid)
        if 0 <= y < grid_mask_bin.shape[0] and 0 <= x < grid_mask_bin.shape[1]:
            if grid_mask_bin[y, x] > 0:
                filtered_pts.append((x, y))

    pts = filtered_pts
    pts_np = np.array(pts, dtype=np.int32)

    # === Save clean points for Stufe 4 PRO ===
    np.save(folder / "07_points.npy", pts_np)
    print(" Saved:", folder / "07_points.npy")

    # ======================================================

    # Visualization for checking
    warp_img = cv2.imread(str(folder / "02_warp_maskprep.jpg"))
    vis = warp_img.copy()

    for (x, y) in pts:
        cv2.circle(vis, (x, y), 5, (0, 0, 255), -1)

    show("Perfect grid intersection points (PRO)", vis)

    cv2.imwrite(str(folder / "07_intersections.jpg"), vis)
    print(" Saved:", folder / "07_intersections.jpg")

print("\n Stufe 3.5 PRO – fertig! Nur echte Gitterpunkte bleiben übrig.")


In [ ]:
# %% =====new Stufe 4 PRO: Build Perfect Uniform Grid from Stufe 3.5 =====


BASE_WARP = Path("DS project/results_warp")
OUT_GRID  = Path("DS project/results_grid")
OUT_GRID.mkdir(parents=True, exist_ok=True)

def show_and_save(title, img, path):
    plt.figure(figsize=(8, 8))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()
    cv2.imwrite(str(path), cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    print(" Saved:", path)

def compute_uniform_lines(lines_raw):
    lines_raw = np.array(sorted(lines_raw))

    diffs = np.diff(lines_raw)
    if len(diffs)==0:
        return lines_raw, 0

    step = int(np.median(diffs))
    return lines_raw, step


# ========== MAIN LOOP ==========
for img_path in CURRENT_IMAGES:

    name = Path(img_path).stem
    print(f"\n Stufe 4 PRO – Processing: {name}")

    warp_folder = BASE_WARP / name
    warp_img_path = warp_folder / "02_warp_maskprep.jpg"
    pts_path      = warp_folder / "07_points.npy"

    if not warp_img_path.exists() or not pts_path.exists():
        print(" Missing warp or points, skipping.")
        continue

    img = cv2.imread(str(warp_img_path))
    H, W = img.shape[:2]

    # ===== LOAD TRUE POINTS FROM 3.5 =====
    pts = np.load(pts_path)

    xs = sorted(set([int(p[0]) for p in pts]))
    ys = sorted(set([int(p[1]) for p in pts]))

    # ===== Compute uniform lines =====
    x_uniform, cell_w = compute_uniform_lines(xs)
    y_uniform, cell_h = compute_uniform_lines(ys)

    # ===== Draw result =====
    overlay = img.copy()
    for x in x_uniform:
        cv2.line(overlay, (x, 0), (x, H-1), (0, 0, 255), 1)
    for y in y_uniform:
        cv2.line(overlay, (0, y), (W-1, y), (0, 0, 255), 1)

    out_folder = OUT_GRID / name
    out_folder.mkdir(exist_ok=True)

    np.savez(out_folder / "grid_data.npz",
             x_lines=x_uniform,
             y_lines=y_uniform,
             cell_width=cell_w,
             cell_height=cell_h,
             rows=len(y_uniform)-1,
             cols=len(x_uniform)-1)

    show_and_save("Uniform Grid (PRO)", overlay, out_folder / "07_uniform_grid.jpg")

    print(" Cell size:", cell_w, "x", cell_h)
    print(" Grid:", len(y_uniform)-1, "rows,", len(x_uniform)-1, "cols")

print("\n Stufe 4 PRO completed — Correct uniform grids generated!")


In [ ]:
# %% ===== Enhanced Stufe 5 PRO: Perfect Cell Extraction (Guaranteed 13×10) =====


GRID_DIR = Path("DS project/results_grid")

EXPECTED_COLS = 10
EXPECTED_ROWS = 13


# ================= Helper =================
def show_and_save(title, img, path):
    plt.figure(figsize=(8, 8))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()
    cv2.imwrite(str(path), img)
    print(" Saved:", path)


# ---- Merge close lines (fix noisy duplicates) ----
def merge_close_lines(lines, min_dist=5):
    lines = sorted(lines)
    clean = [lines[0]]

    for v in lines[1:]:
        if abs(v - clean[-1]) >= min_dist:
            clean.append(v)

    return np.array(clean)


# ---- Force uniform grid of size N using quantization ----
def force_uniform(lines, expected_n, limit_max):
    lines = np.array(sorted(lines))

    # compute spacing
    diffs = np.diff(lines)
    step = int(np.median(diffs))

    # build new uniform grid
    start = lines[0]
    new = [start]

    for _ in range(expected_n):
        new.append(new[-1] + step)

    # clamp last line
    new[-1] = limit_max

    return np.array(new)


# ---- Build cell rectangles ----
def build_cells(x_lines, y_lines, pad=1):
    cells = []

    for r in range(len(y_lines)-1):
        for c in range(len(x_lines)-1):
            x0, x1 = x_lines[c], x_lines[c+1]
            y0, y1 = y_lines[r], y_lines[r+1]
            w = (x1 - x0) - 2*pad
            h = (y1 - y0) - 2*pad

            cells.append((r, c, x0+pad, y0+pad, w, h))

    return cells


# ====================== MAIN PROCESS ======================
try:
    CURRENT_IMAGES
except:
    raise ValueError("CURRENT_IMAGES missing!")

for img_path in CURRENT_IMAGES:

    name = Path(img_path).stem
    print(f"\n Stufe 5 PRO – Processing: {name}")

    folder = GRID_DIR / name
    data_path = folder / "grid_data.npz"
    img_path = folder / "07_uniform_grid.jpg"

    if not data_path.exists() or not img_path.exists():
        print(f"⚠ Missing grid data for {name}. Skipped.")
        continue

    data = np.load(data_path)
    x_raw = list(data["x_lines"])
    y_raw = list(data["y_lines"])

    img = cv2.imread(str(img_path))
    H, W = img.shape[:2]

    # --- 1) merge close duplicates ---
    x_clean = merge_close_lines(x_raw, min_dist=6)
    y_clean = merge_close_lines(y_raw, min_dist=6)

    # --- 2) force exact grid size ---
    x_fixed = force_uniform(x_clean, EXPECTED_COLS, W)
    y_fixed = force_uniform(y_clean, EXPECTED_ROWS, H)

    # --- 3) extract cells ---
    cells = build_cells(x_fixed, y_fixed)

    # --- 4) visualization ---
    vis = img.copy()
    for (_, _, x, y, w, h) in cells:
        cv2.rectangle(vis, (x, y), (x+w, y+h), (0,255,0), 1)

    out_path = folder / "08_cells.jpg"
    show_and_save("Zellen innerhalb des Rasters (PRO)", vis, out_path)

    print(" Rows:", len(y_fixed)-1,
          "| Cols:", len(x_fixed)-1,
          "| Total cells:", len(cells))
    print(" Expected:", EXPECTED_ROWS, "rows ×", EXPECTED_COLS, "cols")

print("\n Enhanced Stufe 5 PRO completed!")


In [ ]:
# %% ===== Stufe 5.1 MANUAL FINAL — Mask by GRID ROW/COL =====


WARP_DIR = Path("DS project/results_warp")
GRID_DIR = Path("DS project/results_grid")

print("\n  Stufe 5.5 MANUAL FINAL — Mask by GRID ROW/COL\n")

# ========
START_ROW = 8
START_COL = 6
END_ROW   = 12
END_COL   = 9
# ==========================

for img_path in CURRENT_IMAGES:

    name = Path(img_path).stem
    print(f"\n Processing: {name}")

    warp_img_path = WARP_DIR / name / "02c_grid_rectified.jpg"
    grid_npz_path = GRID_DIR / name / "grid_data.npz"

    if not warp_img_path.exists():
        print(f" Missing Auto-Warp image for {name}")
        continue
    if not grid_npz_path.exists():
        print(f" Missing grid_data.npz")
        continue

    # ---- Load image + grid ----
    img = cv2.imread(str(warp_img_path))
    data = np.load(grid_npz_path)
    x_lines = data["x_lines"]
    y_lines = data["y_lines"]

    H, W = img.shape[:2]

    # ---- Safe bounds ----
    max_row = len(y_lines) - 2
    max_col = len(x_lines) - 2

    sr = min(START_ROW, max_row)
    er = min(END_ROW, max_row)
    sc = min(START_COL, max_col)
    ec = min(END_COL, max_col)

    # ---- Convert ROW/COL → Pixel ----
    x1 = x_lines[sc]
    x2 = x_lines[ec + 1]
    y1 = y_lines[sr]
    y2 = y_lines[er + 1]

    print(f" Mask area (ROW/COL): R{sr}–R{er}, C{sc}–C{ec}")
    print(f" Converted to pixels: X({x1}:{x2})  Y({y1}:{y2})")

    # ---- Draw grid for preview ----
    preview = img.copy()
    for x in x_lines:
        cv2.line(preview, (x, 0), (x, H), (0, 0, 255), 1)
    for y in y_lines:
        cv2.line(preview, (0, y), (W, y), (0, 0, 255), 1)

    plt.figure(figsize=(8, 8))
    plt.imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
    plt.title("Auto-Warp + GRID (Preview)")
    plt.axis("off")
    plt.show()

    # ---- Apply mask ----
    final = img.copy()
    cv2.rectangle(final, (x1, y1), (x2, y2), (180,180,180), -1)
    blurred = cv2.GaussianBlur(final[y1:y2, x1:x2], (35, 35), 0)
    final[y1:y2, x1:x2] = blurred

    # ---- Show result ----
    plt.figure(figsize=(8, 8))
    plt.imshow(cv2.cvtColor(final, cv2.COLOR_BGR2RGB))
    plt.title("Final MASK using GRID ROW/COL")
    plt.axis("off")
    plt.show()

    # ---- Save ----
    out_path = WARP_DIR / name / f"{name}_masked_grid.jpg"
    cv2.imwrite(str(out_path), final)

    print(f" Saved → {out_path.resolve()}")

print("\n DONE — Manual mask applied using GRID ROW/COL!")


In [ ]:
# %% ===== Stufe 6 — Use masked_grid.jpg + correct coordinates =====


GRID_DIR = Path("DS project/results_grid")
WARP_DIR = Path("DS project/results_warp")

try:
    CURRENT_IMAGES
except NameError:
    raise ValueError("⚠ CURRENT_IMAGES not defined!")

if len(CURRENT_IMAGES) == 0:
    raise ValueError("⚠ No new images uploaded!")

# ------------------------------------------------------
# Remove grid lines (same as before)
# ------------------------------------------------------
def remove_grid_lines(bw):
    h, w = bw.shape
    kx = max(5, int(w * 0.55))
    ky = max(5, int(h * 0.55))

    hor = cv2.morphologyEx(bw, cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_RECT, (kx, 1)))
    ver = cv2.morphologyEx(bw, cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_RECT, (1, ky)))

    cleaned = cv2.subtract(bw, cv2.bitwise_or(hor, ver))
    return cv2.morphologyEx(cleaned, cv2.MORPH_OPEN, np.ones((2,2), np.uint8))

# ------------------------------------------------------
# Text Score
# ------------------------------------------------------
def text_score(roi_bgr):

    g = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2GRAY)
    g = cv2.normalize(g, None, 0, 255, cv2.NORM_MINMAX)

    h, w = g.shape
    area = float(h * w)

    bw = cv2.threshold(
        cv2.GaussianBlur(g, (3,3), 0),
        0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )[1]

    b = max(1, int(0.06 * min(h,w)))
    bw[:b,:] = bw[-b:,:] = bw[:, :b] = bw[:, -b:] = 0

    bw2 = remove_grid_lines(bw)

    ink = bw2.sum() / (255.0 * area)

    cnts, _ = cv2.findContours(bw2, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    small = sum(1 for c in cnts if area * 0.0006 <= cv2.contourArea(c) <= area * 0.08)

    sx = cv2.Sobel(g, cv2.CV_16S, 1,0)
    sy = cv2.Sobel(g, cv2.CV_16S, 0,1)
    mag = cv2.convertScaleAbs(abs(sx)+abs(sy))
    edge_frac = (mag > 40).sum() / area

    score = (
        0.55 * min(1.0, ink / 0.05) +
        0.25 * min(1.0, small / 8.0) +
        0.20 * min(1.0, edge_frac / 0.08)
    )

    return score, ink, small


# ------------------------------------------------------
# MAIN LOOP (UPDATED)
# ------------------------------------------------------
for img_path in CURRENT_IMAGES:

    name = Path(img_path).stem
    print(f"\n Stufe 6 PRO MEGA — processing: {name}")

    # Load grid coordinates
    grid_folder = GRID_DIR / name
    data_path = grid_folder / "grid_data.npz"

    # Load the correct image (UPDATED)
    masked_img_path = WARP_DIR / name / f"{name}_masked_grid.jpg"

    if not data_path.exists():
        print(f"⚠ Missing grid_data for {name}, skipping")
        continue

    if not masked_img_path.exists():
        print(f"⚠ Missing masked_grid.jpg for {name}, skipping")
        continue

    data = np.load(data_path)
    x_lines = data["x_lines"]
    y_lines = data["y_lines"]

    # Load masked grid image (UPDATED)
    img = cv2.imread(str(masked_img_path))

    rows = len(y_lines)-1
    cols = len(x_lines)-1

    results = []
    ink_map = np.zeros((rows, cols), np.float32)
    score_map = np.zeros((rows, cols), np.float32)

    # -----------------------------------------
    # Score each cell using coordinates
    # -----------------------------------------
    for r in range(rows):
        for c in range(cols):
            x0, x1 = x_lines[c], x_lines[c+1]
            y0, y1 = y_lines[r], y_lines[r+1]

            roi = img[y0:y1, x0:x1]
            if roi.size == 0:
                continue

            score, ink, small = text_score(roi)

            results.append([r, c, ink, small, score])
            ink_map[r, c] = ink
            score_map[r, c] = score

    # Save scores CSV
    csv_path = grid_folder / "text_scores.csv"
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["row", "col", "ink", "small_cc", "score"])
        w.writerows(results)

    print(f"Saved: {csv_path}")

    # -----------------------------------------
    # Auto threshold
    # -----------------------------------------
    flat_scores = score_map.flatten()
    ACCEPT_TH = max(0.30, np.percentile(flat_scores, 80))

    text_cells = [(r,c) for r in range(rows) for c in range(cols)
                  if score_map[r,c] >= ACCEPT_TH]

    # -----------------------------------------
    # Auto-detect  (0,0)
    # -----------------------------------------
    if (0,0) not in text_cells:
        x0,x1 = x_lines[0], x_lines[1]
        y0,y1 = y_lines[0], y_lines[1]
        roi0 = img[y0:y1, x0:x1]
        _, ink0, small0 = text_score(roi0)

        mean_ink = np.mean(ink_map[ink_map>0]) if np.any(ink_map>0) else 0

        if (small0 >= 2 and ink0 > 0.4*mean_ink) or ink0 > 0.015:
            text_cells.insert(0,(0,0))
            print("Auto-detected  (0,0)")

    # -----------------------------------------
    # Visualization using masked_grid.jpg (UPDATED)
    # -----------------------------------------
    vis = img.copy()
    for (r,c) in text_cells:
        x0,x1 = x_lines[c], x_lines[c+1]
        y0,y1 = y_lines[r], y_lines[r+1]
        cv2.rectangle(vis,(x0,y0),(x1,y1),(0,0,255),2)

    cv2.imwrite(str(grid_folder/"10_textcells_ACCEPTED_only.jpg"), vis)
    np.save(grid_folder/"text_cells_6_1.npy", np.array(text_cells,dtype=object))

    print(f"Accepted cells: {len(text_cells)}")

print("\nStufe 6.1 finished! (UPDATED TO USE MASKED GRID IMAGE)")


In [ ]:
# %% ===== VISUALIZE DETECTED TEXT CELLS (Stufe 6 OUTPUT) =====


GRID_DIR = Path("DS project/results_grid")
WARP_DIR = Path("DS project/results_warp")

try:
    CURRENT_IMAGES
except NameError:
    raise ValueError("⚠ CURRENT_IMAGES not defined!")

for img_path in CURRENT_IMAGES:

    name = Path(img_path).stem
    print(f"\nVisualizing detected cells — {name}")

    grid_folder = GRID_DIR / name

    grid_npz   = grid_folder / "grid_data.npz"
    cells_npy  = grid_folder / "text_cells_6_1.npy"
    img_path2  = WARP_DIR / name / f"{name}_masked_grid.jpg"

    if not grid_npz.exists() or not cells_npy.exists() or not img_path2.exists():
        print("⚠ Missing required files, skipping")
        continue

    # ---------- Load ----------
    data = np.load(grid_npz)
    x_lines, y_lines = data["x_lines"], data["y_lines"]

    text_cells = np.load(cells_npy, allow_pickle=True)
    img = cv2.imread(str(img_path2))

    vis = img.copy()

    # ---------- Draw boxes ----------
    for (r, c) in text_cells:
        x0, x1 = int(x_lines[c]), int(x_lines[c+1])
        y0, y1 = int(y_lines[r]), int(y_lines[r+1])

        cv2.rectangle(vis, (x0, y0), (x1, y1), (0, 0, 255), 2)

        # optional: cell index label
        cv2.putText(
            vis, f"{r},{c}",
            (x0+5, y0+18),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5, (255,0,0), 1, cv2.LINE_AA
        )

    # ---------- Save + Show ----------
    out_path = grid_folder / "10_TEXTCELLS_DEBUG_VISUAL.jpg"
    cv2.imwrite(str(out_path), vis)

    plt.figure(figsize=(9,9))
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title(f"Detected Text Cells — {name}")
    plt.axis("off")
    plt.show()

    print(f"Saved visualization → {out_path}")

print("\nVisualization finished!")


In [ ]:
# %% ============================================
#      Stufe 6.1 FINAL – CENTER-BOX RECOVERY (ANTI-NOISE)
#      - Runs AFTER Stufe 6
#      - Only on cells rejected by 6
#      - Detects HEAVY / THICK ink only
#      - FIXES:
#         (1) remove grid residual lines BEFORE scoring
#         (2) enforce ink spread (not just a blob)
#         (3) component shape constraints (avoid thin lines/corners)
# ============================================



GRID_DIR = Path("DS project/results_grid")
WARP_DIR = Path("DS project/results_warp")

# ---- Center box ----
CENTER_RATIO = 0.44

# ========= MANUAL MASK (SAME AS STUFE 5.5) =========
START_ROW = 8
END_ROW   = 12
START_COL = 6
END_COL   = 9
# ==================================================

# ========= THICK INK PARAMETERS (tunable) =========
INK_RATIO_TH   = 0.035   # ink ratio inside center box
MIN_COMP_AREA  = 18      # minimum component area
MIN_BIG_COMPS  = 1       # must have at least 1 valid component

# spread constraints (anti-blob)
MIN_PIXELS_IN_CENTER = 10
MIN_SPREAD = 0.15        # relative std spread threshold

# component shape constraints (anti-line / anti-corner)
MIN_COMP_W = 4
MIN_COMP_H = 4
# ================================================


# ---------- Helpers ----------
def cell_bounds(r, c, x_lines, y_lines):
    return (
        int(x_lines[c]), int(y_lines[r]),
        int(x_lines[c+1]), int(y_lines[r+1])
    )

def prep_bin(bgr):
    g = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    g = cv2.GaussianBlur(g, (3,3), 0)
    bw = cv2.threshold(
        g, 0, 255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )[1]
    bw = cv2.morphologyEx(
        bw, cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_RECT, (2,2))
    )
    return bw

def remove_grid_residual(bw):
    """
    Remove long horizontal/vertical strokes (grid lines) from a binary cell.
    This is CRITICAL before measuring ink in center_box.
    """
    h, w = bw.shape

    # robust kernels proportional to cell size
    kx = max(10, w // 2)
    ky = max(10, h // 2)

    hor = cv2.morphologyEx(
        bw, cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_RECT, (kx, 1))
    )
    ver = cv2.morphologyEx(
        bw, cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_RECT, (1, ky))
    )

    cleaned = cv2.subtract(bw, cv2.bitwise_or(hor, ver))

    # slight open to remove tiny specks created by subtraction
    cleaned = cv2.morphologyEx(
        cleaned, cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_RECT, (2,2))
    )

    return cleaned

def is_mask_cell(r, c):
    return (START_ROW <= r <= END_ROW) and (START_COL <= c <= END_COL)

def ink_spread_ok(center_box):
    """
    Prevent false positives where only a small blob / corner noise exists.
    Text should have some spread in X and Y (not concentrated).
    """
    ys, xs = np.where(center_box > 0)

    if len(xs) < MIN_PIXELS_IN_CENTER:
        return False

    # relative spread
    spread_x = float(np.std(xs)) / max(1.0, center_box.shape[1])
    spread_y = float(np.std(ys)) / max(1.0, center_box.shape[0])

    return (spread_x >= MIN_SPREAD) and (spread_y >= MIN_SPREAD)

def count_valid_components(center_box):
    """
    Count components that look like real strokes (avoid thin grid residuals).
    """
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(center_box, connectivity=8)

    big_comp_count = 0
    for i in range(1, num_labels):  # 0 background
        area = stats[i, cv2.CC_STAT_AREA]
        w_c  = stats[i, cv2.CC_STAT_WIDTH]
        h_c  = stats[i, cv2.CC_STAT_HEIGHT]

        if area >= MIN_COMP_AREA and w_c >= MIN_COMP_W and h_c >= MIN_COMP_H:
            big_comp_count += 1

    return big_comp_count


# ============================================
# MAIN
# ============================================

for img_path in CURRENT_IMAGES:

    name = Path(img_path).stem
    folder = GRID_DIR / name

    print(f"\nStufe 6.2 FINAL — CENTER BOX recovery (ANTI-NOISE) — {name}")

    grid_npz      = folder / "grid_data.npz"
    masked_p      = WARP_DIR / name / f"{name}_masked_grid.jpg"
    base_cells_p  = folder / "text_cells_6_1.npy"

    if not grid_npz.exists() or not masked_p.exists() or not base_cells_p.exists():
        print("⚠ Missing required inputs, skipping")
        continue

    # ---------- Load ----------
    data = np.load(grid_npz)
    x_lines, y_lines = data["x_lines"], data["y_lines"]

    img = cv2.imread(str(masked_p))
    if img is None:
        print("⚠ Cannot read masked image, skipping")
        continue

    vis = img.copy()

    rows = len(y_lines) - 1
    cols = len(x_lines) - 1

    base_cells = set(map(tuple, np.load(base_cells_p, allow_pickle=True)))

    recovered_cells = []

    # ---------- CELL LOOP ----------
    for r in range(rows):
        for c in range(cols):

            # Skip already accepted in 6.1
            if (r, c) in base_cells:
                continue

            # Exclude manual mask area
            if is_mask_cell(r, c):
                continue

            x0, y0, x1, y1 = cell_bounds(r, c, x_lines, y_lines)
            roi = img[y0:y1, x0:x1]

            if roi.size == 0:
                continue

            bw = prep_bin(roi)
            if bw.sum() == 0:
                continue

            # CRITICAL FIX: remove grid residual BEFORE center box
            bw = remove_grid_residual(bw)
            if bw.sum() == 0:
                continue

            h, w = bw.shape

            # ---------- CENTER BOX ----------
            left_x   = int(w * CENTER_RATIO)
            right_x  = int(w * (1 - CENTER_RATIO))
            top_y    = int(h * CENTER_RATIO)
            bottom_y = int(h * (1 - CENTER_RATIO))

            center_box = bw[top_y:bottom_y, left_x:right_x]
            if center_box.size == 0:
                continue

            # ---------- (A) Ink Ratio ----------
            ink_pixels = int(np.count_nonzero(center_box))
            area = int(center_box.shape[0] * center_box.shape[1])
            ink_ratio = ink_pixels / max(1, area)

            if ink_ratio < INK_RATIO_TH:
                continue

            # FIX: spread check (avoid blob / corner)
            if not ink_spread_ok(center_box):
                continue

            # ---------- (B) Connected Components (THICK ink + shape) ----------
            big_comp_count = count_valid_components(center_box)
            if big_comp_count < MIN_BIG_COMPS:
                continue

            # ---------- ACCEPT ----------
            recovered_cells.append((r, c))

            # Visualization
            cv2.rectangle(vis, (x0, y0), (x1, y1), (0,255,0), 2)

            # draw center box borders for debug
            cv2.rectangle(
                vis,
                (x0+left_x, y0+top_y),
                (x0+right_x, y0+bottom_y),
                (0,255,255),
                1
            )

    # ---------- MERGE RESULTS ----------
    final_cells = sorted(base_cells | set(recovered_cells))

    np.save(folder / "text_cells.npy", np.array(final_cells, dtype=object))
    np.save(folder / "text_cells_6_2.npy", np.array(recovered_cells, dtype=object))

    out_img = folder / "10_TEXT_CENTER_BOX_RECOVERY_FINAL_ANTI_NOISE.jpg"
    cv2.imwrite(str(out_img), vis)

    plt.figure(figsize=(9,9))
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title(f"Stufe 6.2 FINAL – ANTI-NOISE – {name}")
    plt.axis("off")
    plt.show()

    print(f"Base (6.1): {len(base_cells)}")
    print(f"Recovered (6.2): {len(recovered_cells)}")
    print(f"Final text cells: {len(final_cells)}")
    print("Saved:", out_img)

print("\nStufe 6.2 FINAL (ANTI-NOISE) completed successfully!")


In [ ]:
# %% ===== Stufe 7: Visualization of FINAL detected text cells =====



GRID_DIR = Path("DS project/results_grid")
WARP_DIR = Path("DS project/results_warp")

# === Ensure CURRENT_IMAGES exists ===
try:
    CURRENT_IMAGES
except NameError:
    raise ValueError("CURRENT_IMAGES not defined! Please upload images first.")

if len(CURRENT_IMAGES) == 0:
    raise ValueError("No new images to visualize!")

# === Visualization for new images only ===
for img_path in CURRENT_IMAGES:

    name = Path(img_path).stem
    folder = GRID_DIR / name

    print(f"\nVisualizing FINAL text cells – {name}")

    grid_npz   = folder / "grid_data.npz"
    text_p     = folder / "text_cells.npy"
    masked_img = WARP_DIR / name / f"{name}_masked_grid.jpg"

    if not grid_npz.exists() or not text_p.exists() or not masked_img.exists():
        print("Missing required files, skipping")
        continue

    # ---- Load ----
    data = np.load(grid_npz)
    x_lines, y_lines = data["x_lines"], data["y_lines"]

    text_cells = set(map(tuple, np.load(text_p, allow_pickle=True)))

    img = cv2.imread(str(masked_img))
    if img is None:
        print(f"Could not read image: {masked_img}")
        continue

    vis = img.copy()

    # ---- Draw boxes ----
    for (r, c) in text_cells:
        x0 = int(x_lines[c])
        x1 = int(x_lines[c + 1])
        y0 = int(y_lines[r])
        y1 = int(y_lines[r + 1])

        cv2.rectangle(vis, (x0, y0), (x1, y1), (0, 0, 255), 2)

    # ---- Show ----
    plt.figure(figsize=(9, 9))
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title(f"FINAL Detected Text Cells – {name}", fontsize=14, pad=10)
    plt.show()

print("\nStufe 7pro visualization completed successfully!")


In [ ]:
# %% ============================================
#      Stufe 7A-LARGE – LOCATION ONLY (STABLE)
# ============================================



GRID_DIR = Path("DS project/results_grid")
WARP_DIR = Path("DS project/results_warp")

# --------- Helpers ----------
def cell_bounds(r, c, x_lines, y_lines):
    return (
        int(x_lines[c]), int(y_lines[r]),
        int(x_lines[c+1]), int(y_lines[r+1])
    )

# --------- Triangle templates ----------
def tri_template(size=16, direction="right"):
    img = np.zeros((size, size), np.uint8)
    if direction == "right":
        pts = np.array([[2,2], [2,size-2], [size-2,size//2]], np.int32)
    else:
        pts = np.array([[2,2], [size-2,2], [size//2,size-2]], np.int32)
    cv2.fillConvexPoly(img, pts, 255)
    return img

TRI_R = [tri_template(s, "right") for s in (10,12,14,16,18,20,22,24,26,28)]
TRI_D = [tri_template(s, "down")  for s in (10,12,14,16,18,20,22,24,26,28)]

# --------- Preprocessing ----------
def prep_bin(bgr):
    g = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    g = cv2.GaussianBlur(g, (3,3), 0)
    bw = cv2.threshold(
        g, 0, 255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )[1]
    bw = cv2.morphologyEx(
        bw, cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_RECT, (2,2))
    )
    return bw

# --------- Template matching ----------
def tri_match_scores(bw, direction):

    if direction == "right":
        templates = TRI_R
        stripes = [
            bw[:, int(bw.shape[1]*0.45):],
            bw[:, int(bw.shape[1]*0.60):],
        ]
    else:
        templates = TRI_D
        stripes = [
            bw[int(bw.shape[0]*0.45):, :],
            bw[int(bw.shape[0]*0.60):, :],
        ]

    best = 0.0
    for stripe in stripes:
        if stripe.size == 0:
            continue
        for t in templates:
            if stripe.shape[0] < t.shape[0] or stripe.shape[1] < t.shape[1]:
                continue
            best = max(
                best,
                float(cv2.matchTemplate(stripe, t, cv2.TM_CCOEFF_NORMED).max())
            )
    return best

# ============================================
# MAIN LOOP
# ============================================

ARROW_TH = 0.45
INK_MAX_TEXT = 0.40

for img_path in CURRENT_IMAGES:

    name = Path(img_path).stem
    folder = GRID_DIR / name
    print(f"\nDetecting LARGE arrow LOCATIONS – {name}")

    grid_npz    = folder / "grid_data.npz"
    textcells_p = folder / "text_cells.npy"
    masked_p    = WARP_DIR / name / f"{name}_masked_grid.jpg"

    data = np.load(grid_npz)
    x_lines, y_lines = data["x_lines"], data["y_lines"]
    rows, cols = len(y_lines)-1, len(x_lines)-1

    img = cv2.imread(str(masked_p))

    text_cells = {
        (int(r), int(c))
        for (r, c) in np.load(textcells_p, allow_pickle=True)
    }

    gray_full = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, bw_full = cv2.threshold(
        cv2.GaussianBlur(gray_full, (3,3), 0),
        0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    def cell_ink(r, c):
        x0,y0,x1,y1 = cell_bounds(r,c,x_lines,y_lines)
        roi = bw_full[y0:y1, x0:x1]
        return roi.mean()/255.0 if roi.size else 1.0

    arrow_cells = []
    vis = img.copy()

    for r in range(rows):
        for c in range(cols):

            if (r,c) in text_cells:
                continue

            if cell_ink(r,c) > INK_MAX_TEXT:
                continue

            x0,y0,x1,y1 = cell_bounds(r,c,x_lines,y_lines)
            roi = img[y0:y1, x0:x1]
            if roi.size == 0:
                continue

            bw = prep_bin(roi)

            score_r = tri_match_scores(bw, "right")
            score_d = tri_match_scores(bw, "down")

            if max(score_r, score_d) < ARROW_TH:
                continue

            arrow_cells.append({
                "image": name,
                "row": r,
                "col": c
            })

            cv2.rectangle(vis, (x0,y0), (x1,y1), (0,255,0), 2)

    np.save(folder / "arrow_heads_large.npy", np.array(arrow_cells, dtype=object))
    cv2.imwrite(str(folder / "07A_LARGE_LOCATION.jpg"), vis)

    plt.figure(figsize=(9,9))
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title(f"LARGE arrows – LOCATION ONLY – {name}")
    plt.axis("off")
    plt.show()

    print(f"Detected LARGE arrow cells: {len(arrow_cells)}")

print("\nStufe 7A-LARGE LOCATION stage completed!")

In [ ]:
# %% ============================================
#      Stufe 7A-LARGE – LOCATION ONLY (FINAL FIX)
# ============================================



GRID_DIR = Path("DS project/results_grid")
WARP_DIR = Path("DS project/results_warp")

# --------- Helpers ----------
def cell_bounds(r, c, x_lines, y_lines):
    return (
        int(x_lines[c]), int(y_lines[r]),
        int(x_lines[c+1]), int(y_lines[r+1])
    )

# --------- Triangle templates ----------
def tri_template(size=16, direction="right"):
    img = np.zeros((size, size), np.uint8)
    if direction == "right":
        pts = np.array([[2, 2], [2, size-2], [size-2, size//2]], np.int32)
    else:  # down
        pts = np.array([[2, 2], [size-2, 2], [size//2, size-2]], np.int32)
    cv2.fillConvexPoly(img, pts, 255)
    return img

TRI_R = [tri_template(s, "right") for s in (10,12,14,16,18,20,22,24,26,28)]
TRI_D = [tri_template(s, "down")  for s in (10,12,14,16,18,20,22,24,26,28)]

# --------- Preprocessing ----------
def prep_bin(bgr):
    g = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    g = cv2.GaussianBlur(g, (3,3), 0)
    bw = cv2.threshold(
        g, 0, 255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )[1]

    bw = cv2.morphologyEx(
        bw, cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_RECT, (2,2))
    )
    return bw

# --------- NEW: remove tiny connected components inside a cell ----------
def remove_small_components(bw, min_area):
    """
    bw: binary (0/255) where ink=255
    removes connected components with area < min_area
    """
    num, labels, stats, _ = cv2.connectedComponentsWithStats(bw, connectivity=8)
    if num <= 1:
        return bw

    out = np.zeros_like(bw)
    for i in range(1, num):
        area = stats[i, cv2.CC_STAT_AREA]
        if area >= min_area:
            out[labels == i] = 255
    return out

# --------- Template matching ----------
def tri_match_scores(bw, direction):

    if direction == "right":
        templates = TRI_R
        stripes = [
            bw[:, int(bw.shape[1]*0.45):],
            bw[:, int(bw.shape[1]*0.60):],
        ]
    else:  # down
        templates = TRI_D
        stripes = [
            bw[int(bw.shape[0]*0.45):, :],
            bw[int(bw.shape[0]*0.60):, :],
        ]

    best = 0.0
    for stripe in stripes:
        if stripe.size == 0:
            continue
        for t in templates:
            if stripe.shape[0] < t.shape[0] or stripe.shape[1] < t.shape[1]:
                continue
            best = max(
                best,
                float(cv2.matchTemplate(stripe, t, cv2.TM_CCOEFF_NORMED).max())
            )
    return best

# ============================================
# MAIN LOOP
# ============================================

ARROW_TH = 0.38

# text cells already filtered by your text-cells stage + ink threshold
INK_MAX_TEXT = 0.40

# ink range AFTER cleaning:
INK_CLEAN_MIN = 0.010
INK_CLEAN_MAX = 0.220

for img_path in CURRENT_IMAGES:

    name = Path(img_path).stem
    folder = GRID_DIR / name
    print(f"\n Detecting LARGE arrow LOCATIONS – {name}")

    grid_npz    = folder / "grid_data.npz"
    textcells_p = folder / "text_cells.npy"
    masked_p    = WARP_DIR / name / f"{name}_masked_grid.jpg"

    data = np.load(grid_npz)
    x_lines, y_lines = data["x_lines"], data["y_lines"]
    rows, cols = len(y_lines)-1, len(x_lines)-1

    img = cv2.imread(str(masked_p))

    text_cells = {
        (int(r), int(c))
        for (r, c) in np.load(textcells_p, allow_pickle=True)
    }

    gray_full = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, bw_full = cv2.threshold(
        cv2.GaussianBlur(gray_full, (3,3), 0),
        0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    def cell_ink(r, c):
        x0,y0,x1,y1 = cell_bounds(r,c,x_lines,y_lines)
        roi = bw_full[y0:y1, x0:x1]
        return roi.mean()/255.0 if roi.size else 1.0

    arrow_cells = []
    vis = img.copy()

    for r in range(rows):
        for c in range(cols):

            if (r,c) in text_cells:
                continue

            if cell_ink(r,c) > INK_MAX_TEXT:
                continue

            x0,y0,x1,y1 = cell_bounds(r,c,x_lines,y_lines)
            roi = img[y0:y1, x0:x1]
            if roi.size == 0:
                continue

            bw = prep_bin(roi)

            # ---------- NEW: clean noise inside the cell ----------
            min_area = max(18, int(0.0020 * bw.size))
            bw_clean = remove_small_components(bw, min_area=min_area)

            # ---------- NEW: ink-range after cleaning ----------
            ink_clean = (bw_clean.mean() / 255.0)
            if ink_clean < INK_CLEAN_MIN or ink_clean > INK_CLEAN_MAX:
                continue

            # ---------- template matching (same as original) ----------
            score_r = tri_match_scores(bw_clean, "right")
            score_d = tri_match_scores(bw_clean, "down")

            if max(score_r, score_d) < ARROW_TH:
                continue

            arrow_cells.append({
                "image": name,
                "row": r,
                "col": c
            })

            cv2.rectangle(vis, (x0,y0), (x1,y1), (0,255,0), 2)

    np.save(folder / "arrow_heads_large.npy", np.array(arrow_cells, dtype=object))
    cv2.imwrite(str(folder / "07A_LARGE_LOCATION.jpg"), vis)

    plt.figure(figsize=(9,9))
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title(f"LARGE arrows – LOCATION ONLY – {name}")
    plt.axis("off")
    plt.show()

    print(f"Detected LARGE arrow cells: {len(arrow_cells)}")

print("\nStufe 7A-LARGE LOCATION stage completed!")


In [ ]:
# %% ============================================
#      Stufe 7A-LARGE-2 – DIRECTION ASSIGNMENT
#      - Uses results of 7A-LARGE
#      - BORDER INK (validated)
#      - Updates arrow_heads.npy with direction
#      - Visualization includes split lines
# ============================================



GRID_DIR = Path("DS project/results_grid")
WARP_DIR = Path("DS project/results_warp")

# ---------- Helpers ----------
def cell_bounds(r, c, x_lines, y_lines):
    return (
        int(x_lines[c]), int(y_lines[r]),
        int(x_lines[c+1]), int(y_lines[r+1])
    )

def prep_bin(bgr):
    g = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    g = cv2.GaussianBlur(g, (3,3), 0)
    bw = cv2.threshold(
        g, 0, 255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )[1]
    bw = cv2.morphologyEx(
        bw, cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_RECT, (2,2))
    )
    return bw

# ============================================
# MAIN
# ============================================

RIGHT_STRIP_RATIO  = 0.50   # validated
BOTTOM_STRIP_RATIO = 0.60   # validated

for img_path in CURRENT_IMAGES:

    name = Path(img_path).stem
    folder = GRID_DIR / name

    print(f"\n Assigning LARGE arrow DIRECTIONS – {name}")

    grid_npz  = folder / "grid_data.npz"
    arrows_p  = folder / "arrow_heads_large.npy"
    masked_p  = WARP_DIR / name / f"{name}_masked_grid.jpg"

    if not arrows_p.exists():
        print("⚠ arrow_heads.npy not found, skipping")
        continue

    data = np.load(grid_npz)
    x_lines, y_lines = data["x_lines"], data["y_lines"]

    arrows = np.load(arrows_p, allow_pickle=True)
    img = cv2.imread(str(masked_p))

    results = []
    vis = img.copy()

    for a in arrows:

        r, c = int(a["row"]), int(a["col"])
        x0, y0, x1, y1 = cell_bounds(r, c, x_lines, y_lines)

        roi = img[y0:y1, x0:x1]
        if roi.size == 0:
            continue

        bw = prep_bin(roi)
        h, w = bw.shape

        # ---------- Border strips ----------
        right_strip  = bw[:, int(w * (1 - RIGHT_STRIP_RATIO)):]
        bottom_strip = bw[int(h * (1 - BOTTOM_STRIP_RATIO)):, :]

        ink_right  = np.mean(right_strip == 255)
        ink_bottom = np.mean(bottom_strip == 255)

        # ---------- FINAL DECISION ----------
        if ink_right > ink_bottom:
            direction = "down"
            color = (255, 0, 0)   # Blue = Down
        else:
            direction = "right"
            color = (0, 255, 0)   # Green = Right

        results.append({
            "image": name,
            "row": r,
            "col": c,
            "direction": direction
        })

        # ---------- Visualization ----------
        cv2.rectangle(vis, (x0, y0), (x1, y1), color, 2)

        split_x = x0 + int(w * (1 - RIGHT_STRIP_RATIO))
        split_y = y0 + int(h * (1 - BOTTOM_STRIP_RATIO))

        # Vertical split (right region)
        cv2.line(
            vis,
            (split_x, y0),
            (split_x, y1),
            (0, 255, 255),  # Yellow
            1
        )

        # Horizontal split (bottom region)
        cv2.line(
            vis,
            (x0, split_y),
            (x1, split_y),
            (255, 255, 0),  # Cyan
            1
        )

    # -------- overwrite arrow_heads.npy with directions --------
    np.save(folder / "arrow_heads_large_dir.npy", np.array(results, dtype=object))
    cv2.imwrite(str(folder / "07A_LARGE_2_DIRECTION.jpg"), vis)

    plt.figure(figsize=(9,9))
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title(f"LARGE arrows – LOCATION + DIRECTION – {name}")
    plt.axis("off")
    plt.show()

    print(f"Directions assigned: {len(results)}")

print("\nStufe 7A-LARGE-2 completed successfully!")


In [ ]:
# %% ============================================
#      Stufe 7A SMALL – Arrow HEAD detection
# ============================================


GRID_DIR = Path("DS project/results_grid")
WARP_DIR = Path("DS project/results_warp")

def cell_bounds(r, c, x_lines, y_lines):
    return int(x_lines[c]), int(y_lines[r]), int(x_lines[c+1]), int(y_lines[r+1])

def tri_template(size=16, direction="right"):
    k = np.zeros((size, size), np.uint8)
    if direction == "right":
        pts = np.array([[2,2],[2,size-2],[size-2,size//2]], np.int32)
    else:
        pts = np.array([[2,2],[size-2,2],[size//2,size-2]], np.int32)
    cv2.fillConvexPoly(k, pts, 255)
    return k

TRI_R = [tri_template(s,"right") for s in (10,12,14,16,18,20,22,24,26,28)]
TRI_D = [tri_template(s,"down")  for s in (10,12,14,16,18,20,22,24,26,28)]

def prep_bin(bgr):
    g = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    g = cv2.GaussianBlur(g,(3,3),0)
    bw = cv2.threshold(g,0,255,cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)[1]
    bw = cv2.morphologyEx(
        bw, cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_RECT,(2,2))
    )
    return bw

def tri_match_scores(bw, direction):
    if direction == "right":
        templates = TRI_R
        stripes = [bw[:, int(bw.shape[1]*0.45):], bw[:, int(bw.shape[1]*0.60):]]
    else:
        templates = TRI_D
        stripes = [bw[int(bw.shape[0]*0.45):, :], bw[int(bw.shape[0]*0.60):, :]]

    best = 0
    for stripe in stripes:
        if stripe.size == 0: continue
        for t in templates:
            if stripe.shape[0] < t.shape[0] or stripe.shape[1] < t.shape[1]: continue
            best = max(best, float(cv2.matchTemplate(stripe, t, cv2.TM_CCOEFF_NORMED).max()))

    for t in templates:
        if bw.shape[0] < t.shape[0] or bw.shape[1] < t.shape[1]: continue
        best = max(best, float(cv2.matchTemplate(bw, t, cv2.TM_CCOEFF_NORMED).max()))

    return best

# ============================================
# MAIN LOOP
# ============================================

ARROW_TH = 0.45
INK_MAX_TEXT = 0.40

for img_path in CURRENT_IMAGES:

    name = Path(img_path).stem
    folder = GRID_DIR / name

    print(f"\n Stufe 7A SMALL – detecting for: {name}")

    large_file = folder / "arrow_heads_large.npy"
    if large_file.exists():
        large_data = np.load(large_file, allow_pickle=True).tolist()
    else:
        large_data = []

    large_set = {(int(a["row"]), int(a["col"])) for a in large_data}

    grid_npz = folder / "grid_data.npz"
    textcells_p = folder / "text_cells.npy"
    masked_grid_img = WARP_DIR / name / f"{name}_masked_grid.jpg"

    data = np.load(grid_npz)
    x_lines, y_lines = data["x_lines"], data["y_lines"]
    rows, cols = len(y_lines)-1, len(x_lines)-1

    img = cv2.imread(str(masked_grid_img))

    text_cells = {(int(r), int(c)) for (r,c) in np.load(textcells_p, allow_pickle=True)}

    gray_full = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, bw_full = cv2.threshold(
        cv2.GaussianBlur(gray_full,(3,3),0),
        0,255,cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU
    )

    def cell_ink(r,c):
        x0,y0,x1,y1 = cell_bounds(r,c,x_lines,y_lines)
        roi = bw_full[y0:y1, x0:x1]
        return roi.mean()/255.0 if roi.size else 1.0

    small_arrows = []
    vis = img.copy()

    for r in range(rows):
        for c in range(cols):

            if (r,c) in text_cells: continue
            if (r,c) in large_set: continue
            if cell_ink(r,c) > INK_MAX_TEXT: continue

            x0,y0,x1,y1 = cell_bounds(r,c,x_lines,y_lines)
            roi = img[y0:y1, x0:x1]

            bw = prep_bin(roi)
            score_r = tri_match_scores(bw,"right")
            score_d = tri_match_scores(bw,"down")

            dirs = []
            if score_r >= ARROW_TH: dirs.append("right")
            if score_d >= ARROW_TH: dirs.append("down")
            if not dirs: continue

            color = (0,255,0) if len(dirs)==1 else (0,165,255)
            cv2.rectangle(vis,(x0,y0),(x1,y1),color,2)

            small_arrows.append({
                "image": name,
                "row": r,
                "col": c,
                "directions": dirs,
                "score": float(max(score_r, score_d))
            })


    np.save(folder / "arrow_heads_small.npy", np.array(small_arrows, dtype=object))

    cv2.imwrite(str(folder / "07A_SMALL_ONLY_VIS.jpg"), vis)

    plt.figure(figsize=(9,9))
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title(f"SMALL arrows – {name}")
    plt.axis("off")
    plt.show()

    print(f"SMALL arrows detected: {len(small_arrows)}")

print("\nStufe 7A SMALL completed (NO MERGE).")


In [ ]:
# ============================================
#      Stufe 7B — Question + Arrow HEAD viz
#      (LARGE + SMALL separated)
# ============================================



GRID_DIR = Path("DS project/results_grid")
WARP_DIR = Path("DS project/results_warp")

# -----------------------------
# Ensure CURRENT_IMAGES
# -----------------------------
try:
    CURRENT_IMAGES
except NameError:
    raise ValueError(" CURRENT_IMAGES not defined!")

TARGET_FOLDERS = {Path(p).stem for p in CURRENT_IMAGES}
print(" Processing only:", TARGET_FOLDERS)

# -----------------------------
# Helpers
# -----------------------------
def cell_bounds(r, c, x_lines, y_lines):
    return int(x_lines[c]), int(y_lines[r]), int(x_lines[c+1]), int(y_lines[r+1])

# -----------------------------
# MAIN LOOP
# -----------------------------
for folder in GRID_DIR.iterdir():

    if not folder.is_dir() or folder.name not in TARGET_FOLDERS:
        continue

    print(f"\n Visualizing Q + LARGE + SMALL arrows for: {folder.name}")

    grid_npz     = folder / "grid_data.npz"
    text_cells_p = folder / "text_cells.npy"
    large_p      = folder / "arrow_heads_large.npy"
    small_p      = folder / "arrow_heads_small.npy"

    masked_img_p = WARP_DIR / folder.name / f"{folder.name}_masked_grid.jpg"

    if not (grid_npz.exists() and text_cells_p.exists() and masked_img_p.exists()):
        print("⚠ Missing required files, skipping.")
        continue

    data = np.load(grid_npz)
    x_lines, y_lines = data["x_lines"], data["y_lines"]

    img = cv2.imread(str(masked_img_p))
    vis = img.copy()

    text_cells = np.load(text_cells_p, allow_pickle=True).tolist()

    large_arrows = np.load(large_p, allow_pickle=True).tolist() if large_p.exists() else []
    small_arrows = np.load(small_p, allow_pickle=True).tolist() if small_p.exists() else []

    # -----------------------------
    # Draw QUESTIONS (RED)
    # -----------------------------
    for i, (r, c) in enumerate(text_cells):
        x0,y0,x1,y1 = cell_bounds(int(r), int(c), x_lines, y_lines)
        cv2.rectangle(vis,(x0,y0),(x1,y1),(0,0,255),2)
        cv2.putText(vis,f"Q{i}",(x0+4,y0+16),
                    cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,0,255),1)

    # -----------------------------
    # Draw LARGE arrows (GREEN)
    # -----------------------------
    for j,a in enumerate(large_arrows):
        r,c = int(a["row"]), int(a["col"])
        x0,y0,x1,y1 = cell_bounds(r,c,x_lines,y_lines)
        cv2.rectangle(vis,(x0,y0),(x1,y1),(0,255,0),2)
        cv2.putText(vis,f"L{j}",(x0+4,y0+16),
                    cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,255,0),1)

    # -----------------------------
    # Draw SMALL arrows (BLUE)
    # -----------------------------
    for k,a in enumerate(small_arrows):
        r,c = int(a["row"]), int(a["col"])
        x0,y0,x1,y1 = cell_bounds(r,c,x_lines,y_lines)
        cv2.rectangle(vis,(x0,y0),(x1,y1),(255,0,0),2)
        cv2.putText(vis,f"S{k}",(x0+4,y0+16),
                    cv2.FONT_HERSHEY_SIMPLEX,0.5,(255,0,0),1)

    out_path = folder / "07B_Q_LARGE_SMALL_VIS.jpg"
    cv2.imwrite(str(out_path), vis)

    plt.figure(figsize=(10,10))
    plt.imshow(cv2.cvtColor(vis,cv2.COLOR_BGR2RGB))
    plt.title(f"Questions (red) | Large (green) | Small (blue) – {folder.name}")
    plt.axis("off")
    plt.show()

    print("Saved →", out_path)

print("\n Stufe 7B completed (LARGE + SMALL).")


In [ ]:
# %% ===== Stufe 8 PRO — Final Answer Assignment (MULTI-ARROW SAFE) =====


GRID_DIR = Path("DS project/results_grid")

try:
    CURRENT_IMAGES
except NameError:
    raise ValueError("⚠ CURRENT_IMAGES missing!")

if len(CURRENT_IMAGES) == 0:
    raise ValueError("⚠ No images found!")


def euclidean(p1, p2):
    return float(np.sqrt((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2))


# =====================================================================
# MAIN LOOP
# =====================================================================
for img_path in CURRENT_IMAGES:

    name = Path(img_path).stem
    folder = GRID_DIR / name
    print(f"\n Stufe 8 — Processing: {name}")

    grid_npz      = folder / "grid_data.npz"
    text_cells_p  = folder / "text_cells.npy"
    arrow_heads_p = folder / "arrow_heads.npy"

    if not (grid_npz.exists() and text_cells_p.exists() and arrow_heads_p.exists()):
        print(f"⚠ Missing files for {name}, skipping.")
        continue

    # -------------------------------------------------
    # Load data
    # -------------------------------------------------
    data = np.load(grid_npz)
    x_lines, y_lines = data["x_lines"], data["y_lines"]

    text_cells  = np.load(text_cells_p, allow_pickle=True).tolist()
    arrow_heads = np.load(arrow_heads_p, allow_pickle=True).tolist()

    if len(text_cells) == 0 or len(arrow_heads) == 0:
        print("⚠ No questions or arrows found, skipping.")
        continue

    # -------------------------------------------------
    # EXPLODE arrows (IMPORTANT FIX)
    # -------------------------------------------------
    arrow_instances = []

    for a in arrow_heads:
        r = int(a["row"])
        c = int(a["col"])

        # LARGE or single-arrow format
        if "direction" in a:
            arrow_instances.append({
                "row": r,
                "col": c,
                "direction": a["direction"]
            })

        # SMALL arrows (possibly 2 arrows in one cell)
        elif "directions" in a:
            for d in a["directions"]:
                arrow_instances.append({
                    "row": r,
                    "col": c,
                    "direction": d
                })

    QT = len(text_cells)
    AT = len(arrow_instances)

    print(f"Questions: {QT} | Arrow instances: {AT}")

    if AT == 0:
        print("⚠ No valid arrow instances found.")
        continue

    # =====================================================================
    # STEP 1 — cell centers
    # =====================================================================
    def cell_center(r, c):
        x0, x1 = x_lines[c], x_lines[c+1]
        y0, y1 = y_lines[r], y_lines[r+1]
        return ((x0+x1)//2, (y0+y1)//2)

    Q_centers = {
        i: cell_center(r, c)
        for i, (r, c) in enumerate(text_cells)
    }

    A_centers = {
        i: cell_center(a["row"], a["col"])
        for i, a in enumerate(arrow_instances)
    }

    # =====================================================================
    # STEP 2 — nearest arrows for each question
    # =====================================================================
    nearest = {}

    for qi in range(QT):
        qc = Q_centers[qi]
        dlist = [(euclidean(qc, A_centers[ai]), ai) for ai in range(AT)]
        dlist.sort(key=lambda x: x[0])

        # allow 2 answers if almost equal distance
        if len(dlist) > 1 and abs(dlist[0][0] - dlist[1][0]) < 2.0:
            nearest[qi] = [dlist[0][1], dlist[1][1]]
        else:
            nearest[qi] = [dlist[0][1]]

    # =====================================================================
    # STEP 3 — direct assignment
    # =====================================================================
    assigned_Q = {}
    assigned_A = set()
    conflicts = []

    for qi, candidates in nearest.items():
        if len(candidates) == 1:
            ai = candidates[0]
            if ai not in assigned_A:
                assigned_Q[qi] = ai
                assigned_A.add(ai)
            else:
                conflicts.append(qi)
        else:
            conflicts.append(qi)

    # =====================================================================
    # STEP 4 — conflict resolution
    # =====================================================================
    free_answers = set(range(AT)) - assigned_A

    # Priority 1: only one free option
    for qi in conflicts[:]:
        options = [ai for ai in nearest[qi] if ai in free_answers]
        if len(options) == 1:
            ai = options[0]
            assigned_Q[qi] = ai
            assigned_A.add(ai)
            free_answers.remove(ai)
            conflicts.remove(qi)

    # Priority 2: nearest free
    for qi in conflicts:
        if not free_answers:
            break

        options = [ai for ai in nearest[qi] if ai in free_answers]
        ai = options[0] if options else next(iter(free_answers))

        assigned_Q[qi] = ai
        assigned_A.add(ai)
        free_answers.remove(ai)

    # =====================================================================
    # STEP 5 — OUTPUT
    # =====================================================================
    print("\nFINAL QUESTION → ANSWER MAPPING")
    print("-----------------------------------")

    result_pairs = []

    for qi, ai in assigned_Q.items():
        qr, qc = text_cells[qi]
        ar = arrow_instances[ai]["row"]
        ac = arrow_instances[ai]["col"]
        direction = arrow_instances[ai]["direction"]

        print(
            f"Q{qi} ({qr},{qc})  →  "
            f"A{ai} ({ar},{ac})  direction={direction}"
        )

        result_pairs.append([
            qi, (qr, qc),
            ai, (ar, ac),
            direction
        ])

    out_csv = folder / "final_question_answer_pairs.csv"
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow([
            "Q_index",
            "Q_cell(row,col)",
            "A_index",
            "A_cell(row,col)",
            "direction"
        ])
        w.writerows(result_pairs)

    print("Saved →", out_csv)

print("\n Stufe 8 PRO finished successfully!")



In [ ]:
# %% ============================================
#      Stufe 8 PRO — FINAL ANSWER ASSIGNMENT
#      PASS 1: Adjacent + Direction  (SMALL ONLY)
#      PASS 2: Grid Walk             (LARGE ONLY)
#      CASE B WITH DOWN-DIAGONALS
# ============================================


GRID_DIR = Path("DS project/results_grid")
WARP_DIR = Path("DS project/results_warp")

# ------------------------------------------------
# Helpers
# ------------------------------------------------
def cell_center(r, c, x_lines, y_lines):
    return int((x_lines[c] + x_lines[c+1]) // 2), int((y_lines[r] + y_lines[r+1]) // 2)

def euclidean(p1, p2):
    return float(np.sqrt((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2))

def neighbors(r, c, rows, cols):
    cand = [(r, c+1), (r, c-1), (r+1, c), (r-1, c)]
    return [(rr,cc) for rr,cc in cand if 0 <= rr < rows and 0 <= cc < cols]

def directionally_valid(qr, qc, ar, ac, direction):
    if direction == "right":
        return ac > qc
    if direction == "down":
        return ar > qr
    return False

# ------------------------------------------------
# MAIN LOOP
# ------------------------------------------------
for img_path in CURRENT_IMAGES:

    name = Path(img_path).stem
    folder = GRID_DIR / name
    print(f"\n Stufe 8 FINAL — Processing: {name}")

    grid_npz = folder / "grid_data.npz"
    text_p   = folder / "text_cells.npy"
    small_p  = folder / "arrow_heads_small.npy"
    large_p  = folder / "arrow_heads_large_dir.npy"
    img_p    = WARP_DIR / name / f"{name}_masked_grid.jpg"

    if not (grid_npz.exists() and text_p.exists() and small_p.exists() and large_p.exists()):
        print("⚠ Missing files, skipping.")
        continue

    # ---------- load ----------
    data = np.load(grid_npz)
    x_lines, y_lines = data["x_lines"], data["y_lines"]
    rows, cols = len(y_lines)-1, len(x_lines)-1

    text_cells = [tuple(map(int,x)) for x in np.load(text_p, allow_pickle=True)]

    small_raw = np.load(small_p, allow_pickle=True).tolist()
    large_raw = np.load(large_p, allow_pickle=True).tolist()

    img = cv2.imread(str(img_p))
    vis = img.copy()

    # ---------- explode arrows ----------
    small_arrows = []
    for a in small_raw:
        r, c = int(a["row"]), int(a["col"])
        for d in a.get("directions", []):
            small_arrows.append({"row": r, "col": c, "direction": d})

    large_arrows = [
        {"row": int(a["row"]), "col": int(a["col"]), "direction": a["direction"]}
        for a in large_raw
    ]

    # ---------- centers ----------
    Q_centers = {i: cell_center(r,c,x_lines,y_lines) for i,(r,c) in enumerate(text_cells)}
    A_centers_small = {i: cell_center(a["row"],a["col"],x_lines,y_lines) for i,a in enumerate(small_arrows)}
    A_centers_large = {i: cell_center(a["row"],a["col"],x_lines,y_lines) for i,a in enumerate(large_arrows)}

    assigned_Q = {}
    used_small = set()
    used_large = set()

    # ==================================================
    # PASS 1 — ADJACENT + DIRECTION (SMALL ONLY)
    # ==================================================
    for qi,(qr,qc) in enumerate(text_cells):

        cand = []
        for ai,a in enumerate(small_arrows):
            if ai in used_small:
                continue
            ar,ac = a["row"],a["col"]
            if abs(ar-qr)+abs(ac-qc) != 1:
                continue
            if not directionally_valid(qr,qc,ar,ac,a["direction"]):
                continue
            d = euclidean(Q_centers[qi], A_centers_small[ai])
            cand.append((d,ai))

        if cand:
            cand.sort(key=lambda x:x[0])
            ai = cand[0][1]
            assigned_Q[qi] = ("small", ai)
            used_small.add(ai)

    # ==================================================
    # PASS 2 — GRID WALK (LARGE ONLY)
    # ==================================================
    for r in range(rows):
        for c in range(cols):

            # ---- CASE A: unused LARGE arrow ----
            for ai,a in enumerate(large_arrows):
                if ai in used_large:
                    continue
                if (a["row"],a["col"]) != (r,c):
                    continue

                for (nr,nc) in neighbors(r,c,rows,cols):
                    if (nr,nc) in text_cells:
                        qi = text_cells.index((nr,nc))
                        if qi not in assigned_Q:
                            assigned_Q[qi] = ("large", ai)
                            used_large.add(ai)
                            break

            # ---- CASE B: unassigned QUESTION ----
            if (r,c) in text_cells:
                qi = text_cells.index((r,c))
                if qi in assigned_Q:
                    continue

                cand_cells = neighbors(r,c,rows,cols) + [
                    (r+1, c+1),
                    (r+1, c-1)
                ]

                cand_cells = [
                    (rr,cc) for rr,cc in cand_cells
                    if 0 <= rr < rows and 0 <= cc < cols
                ]

                for (nr,nc) in cand_cells:
                    for ai,a in enumerate(large_arrows):
                        if ai in used_large:
                            continue
                        if (a["row"],a["col"]) == (nr,nc):
                            assigned_Q[qi] = ("large", ai)
                            used_large.add(ai)
                            break
                    if qi in assigned_Q:
                        break

    # ==================================================
    # VISUAL + SAVE
    # ==================================================
    results = []

    for qi,(src,ai) in assigned_Q.items():
        qr,qc = text_cells[qi]

        if src == "small":
            a = small_arrows[ai]
        else:
            a = large_arrows[ai]

        ar,ac = a["row"],a["col"]
        d     = a["direction"]

        qx,qy = cell_center(qr,qc,x_lines,y_lines)
        ax,ay = cell_center(ar,ac,x_lines,y_lines)

        cv2.circle(vis,(qx,qy),6,(255,0,0),-1)
        cv2.circle(vis,(ax,ay),6,(0,0,255),-1)
        cv2.line(vis,(qx,qy),(ax,ay),(80,80,80),2)

        label = "R" if d=="right" else "D"
        cv2.putText(vis,label,(ax+6,ay-6),
                    cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,0,0),3)
        cv2.putText(vis,label,(ax+6,ay-6),
                    cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,255,255),1)

        results.append([qi,(qr,qc),(ar,ac),d,src])
        print(f"Q{qi} ({qr},{qc}) → {src.upper()} A ({ar},{ac}) dir={d}")

    csv_p = folder / "final_question_answer_pairs.csv"
    with open(csv_p,"w",newline="",encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["Q_index","Q_cell","A_cell","direction","arrow_type"])
        w.writerows(results)

    out_img = folder / "08_FINAL_QA_PASS1_PASS2.jpg"
    cv2.imwrite(str(out_img),vis)

    plt.figure(figsize=(10,10))
    plt.imshow(cv2.cvtColor(vis,cv2.COLOR_BGR2RGB))
    plt.title(f"FINAL QA — PASS1 (SMALL) + PASS2 (LARGE) — {name}")
    plt.axis("off")
    plt.show()

    print(" Saved:", csv_p)
    print(" Saved:", out_img)

print("\n Stufe 8 FINAL COMPLETED (SMALL → PASS1 | LARGE → PASS2)")


In [ ]:
# %% ============================================
#      Stufe 9 — Answer Length VISUALIZATION
#      (Arrow cell IS first answer cell)
# ============================================


GRID_DIR = Path("DS project/results_grid")
WARP_DIR = Path("DS project/results_warp")

# ===== MASK REGION (FINAL) =====
START_ROW = 8
START_COL = 6
END_ROW   = 12
END_COL   = 9
# ==============================

def in_bounds(r, c, rows, cols):
    return 0 <= r < rows and 0 <= c < cols

def is_mask_cell(r, c):
    return START_ROW <= r <= END_ROW and START_COL <= c <= END_COL

def step_cell(r, c, direction):
    return (r, c + 1) if direction == "right" else (r + 1, c)

def cell_center(r, c, x_lines, y_lines):
    x0, x1 = x_lines[c], x_lines[c+1]
    y0, y1 = y_lines[r], y_lines[r+1]
    return int((x0 + x1) // 2), int((y0 + y1) // 2)

# ============================================
# MAIN
# ============================================

for img_path in CURRENT_IMAGES:

    name = Path(img_path).stem
    folder = GRID_DIR / name
    print(f"\n Stufe 9 — Answer Length (FINAL LOGIC): {name}")

    grid_npz = folder / "grid_data.npz"
    qa_csv   = folder / "final_question_answer_pairs.csv"
    text_p   = folder / "text_cells.npy"
    img_p    = WARP_DIR / name / f"{name}_masked_grid.jpg"

    if not (grid_npz.exists() and qa_csv.exists() and text_p.exists()):
        print("⚠ Missing files, skipping.")
        continue

    # ---------- load ----------
    data = np.load(grid_npz)
    x_lines, y_lines = data["x_lines"], data["y_lines"]
    rows, cols = len(y_lines) - 1, len(x_lines) - 1

    text_cells = set(tuple(x) for x in np.load(text_p, allow_pickle=True))
    img = cv2.imread(str(img_p))
    vis = img.copy()

    with open(qa_csv, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        pairs = list(reader)

    # ============================================
    # PROCESS EACH QUESTION
    # ============================================
    for p in pairs:

        ar, ac = eval(p["A_cell"])
        direction = p["direction"]

        #  start from ARROW cell itself
        cr, cc = ar, ac
        length = 0
        path = []

        while True:

            if not in_bounds(cr, cc, rows, cols):
                break

            #  stop at mask
            if is_mask_cell(cr, cc):
                break

            #  stop at OTHER question cells
            if (cr, cc) in text_cells and (cr, cc) != (ar, ac):
                break

            #  count answer cell (arrow cell included)
            path.append((cr, cc))
            length += 1

            # move forward
            cr, cc = step_cell(cr, cc, direction)

        # ---------- draw ----------
        for (pr, pc) in path:
            cx, cy = cell_center(pr, pc, x_lines, y_lines)
            cv2.circle(vis, (cx, cy), 5, (0, 165, 255), -1)

        # ---------- label ----------
        ax, ay = cell_center(ar, ac, x_lines, y_lines)
        cv2.putText(
            vis, f"L={length}",
            (ax - 14, ay - 12),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (0, 0, 0), 3, cv2.LINE_AA
        )
        cv2.putText(
            vis, f"L={length}",
            (ax - 14, ay - 12),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (255, 255, 0), 1, cv2.LINE_AA
        )

        print(f"Start({ar},{ac}) dir={direction} → length={length}")

    # ---------- save + show ----------
    out_img = folder / "09_ANSWER_LENGTHS_FINAL_ARROW_CELL_INCLUDED.jpg"
    cv2.imwrite(str(out_img), vis)

    plt.figure(figsize=(10, 10))
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title(f"Answer Lengths (Arrow Cell Included) — {name}")
    plt.axis("off")
    plt.show()

    print(" Saved:", out_img)

print("\n Stufe 9 COMPLETED — arrow cell counted correctly")


In [ ]:
# %% ============================================
#      Stufe 10 — FINAL MERGED QA RESULT
#      (Question + Answer + Direction + Length)
# ============================================


GRID_DIR = Path("DS project/results_grid")
WARP_DIR = Path("DS project/results_warp")

# ===== MASK REGION (from Stufe 5.5) =====
START_ROW = 8
START_COL = 6
END_ROW   = 12
END_COL   = 9
# ======================================

def in_bounds(r, c, rows, cols):
    return 0 <= r < rows and 0 <= c < cols

def is_mask_cell(r, c):
    return START_ROW <= r <= END_ROW and START_COL <= c <= END_COL

def step_cell(r, c, direction):
    return (r, c + 1) if direction == "right" else (r + 1, c)

# ============================================
# MAIN
# ============================================

for img_path in CURRENT_IMAGES:

    name = Path(img_path).stem
    folder = GRID_DIR / name
    print(f"\nStufe 10 — Merging final data: {name}")

    grid_npz = folder / "grid_data.npz"
    qa_csv   = folder / "final_question_answer_pairs.csv"
    text_p   = folder / "text_cells.npy"

    if not (grid_npz.exists() and qa_csv.exists() and text_p.exists()):
        print("⚠ Missing files, skipping.")
        continue

    # ---------- load ----------
    data = np.load(grid_npz)
    x_lines, y_lines = data["x_lines"], data["y_lines"]
    rows, cols = len(y_lines) - 1, len(x_lines) - 1

    text_cells = set(tuple(x) for x in np.load(text_p, allow_pickle=True))

    with open(qa_csv, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        pairs = list(reader)

    final_rows = []

    for p in pairs:

        qr, qc = eval(p["Q_cell"])
        ar, ac = eval(p["A_cell"])
        direction = p["direction"]

        # ================================
        # compute answer length (incl arrow)
        # ================================
        cr, cc = ar, ac
        length = 1

        while True:
            cr, cc = step_cell(cr, cc, direction)

            if not in_bounds(cr, cc, rows, cols):
                break
            if is_mask_cell(cr, cc):
                break
            if (cr, cc) in text_cells:
                break

            length += 1

        final_rows.append([
            (qr, qc),
            (ar, ac),
            direction,
            length
        ])

        print(
            f"Q({qr},{qc}) → A({ar},{ac}) dir={direction} length={length}"
        )

    # ---------- save ----------
    out_csv = folder / "FINAL_QA_COMPLETE.csv"
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow([
            "Question_cell(row,col)",
            "Answer_start_cell(row,col)",
            "Direction",
            "Answer_length"
        ])
        w.writerows(final_rows)

    print(" Saved:", out_csv)

print("\n Stufe 10 COMPLETED — All data in one file")


In [ ]:

GRID_DIR = Path("DS project/results_grid")

for img_path in CURRENT_IMAGES:
    name = Path(img_path).stem
    folder = GRID_DIR / name

    csv_path = folder / "FINAL_QA_COMPLETE.csv"

    if not csv_path.exists():
        print(" File not found:", csv_path)
        continue

    df = pd.read_csv(csv_path)

    print(f"\n Results table — {name}")
    display(df)


In [ ]:
os.environ["OPENAI_API_KEY"] = ""YOUR_OPENAI_API_KEY"

In [ ]:
# =========================================================
#   OCR QUESTIONS FROM GRID USING OPENAI VISION (FINAL)
#   - Single file
#   - Correct OpenAI Responses API usage
#   - German crossword clues
# =========================================================


# ===============================
# OPENAI CLIENT
# ===============================
client = OpenAI()  # API key from environment

# ===============================
# CONFIG
# ===============================
GRID_DIR = Path("DS project/results_grid")
WARP_DIR = Path("DS project/results_warp")

# ===============================
# HELPERS
# ===============================
def cell_bbox(r, c, x_lines, y_lines, pad=4):
    x0 = int(x_lines[c]) + pad
    x1 = int(x_lines[c + 1]) - pad
    y0 = int(y_lines[r]) + pad
    y1 = int(y_lines[r + 1]) - pad
    return x0, y0, x1, y1


def encode_image_data_url(img_bgr):
    """
    Encode OpenCV image to base64 DATA URL (required by OpenAI)
    """
    success, buffer = cv2.imencode(".png", img_bgr)
    if not success:
        return None
    b64 = base64.b64encode(buffer).decode("utf-8")
    return f"data:image/png;base64,{b64}"


def ocr_cell_openai(roi_bgr):
    """
    OCR a single crossword question cell using OpenAI Vision
    """
    if roi_bgr is None or roi_bgr.size == 0:
        return ""

    # Upscale for better recognition
    roi = cv2.resize(
        roi_bgr,
        None,
        fx=2.5,
        fy=2.5,
        interpolation=cv2.INTER_CUBIC
    )

    img_data_url = encode_image_data_url(roi)
    if img_data_url is None:
        return ""

    prompt = (
        "You are reading a German crossword puzzle clue.\n"
        "The image contains printed German text inside one crossword cell.\n"
        "The text may span multiple lines and may contain hyphenation.\n"
        "Please transcribe the text accurately.\n"
        "Merge hyphenated words correctly.\n"
        "Preserve punctuation like (ugs.), commas, and abbreviations.\n"
        "Return ONLY the text, without explanations or quotes."
    )

    response = client.responses.create(
        model="gpt-4.1",
        input=[{
            "role": "user",
            "content": [
                {"type": "input_text", "text": prompt},
                {
                    "type": "input_image",
                    "image_url": img_data_url
                }
            ]
        }]
    )

    return response.output_text.strip()


# ===============================
# MAIN PIPELINE
# ===============================
for folder in GRID_DIR.iterdir():

    if not folder.is_dir():
        continue

    name = folder.name
    print(f"\nOpenAI OCR — {name}")
    print("-" * 60)

    grid_npz = folder / "grid_data.npz"
    qa_csv   = folder / "FINAL_QA_COMPLETE.csv"
    img_p    = WARP_DIR / name / "02_warp_maskprep.jpg"

    if not (grid_npz.exists() and qa_csv.exists() and img_p.exists()):
        print("Missing required files")
        continue

    # ---------- load grid ----------
    grid_data = np.load(grid_npz)
    x_lines = grid_data["x_lines"]
    y_lines = grid_data["y_lines"]

    # ---------- load image ----------
    img = cv2.imread(str(img_p))
    if img is None:
        print("Cannot read warped image")
        continue

    # ---------- load QA mapping ----------
    with open(qa_csv, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        qa_rows = list(reader)

    results = []

    # ---------- OCR each question cell ----------
    for row in qa_rows:
        qr, qc = ast.literal_eval(row["Question_cell(row,col)"])

        x0, y0, x1, y1 = cell_bbox(qr, qc, x_lines, y_lines)
        roi = img[y0:y1, x0:x1]

        text = ocr_cell_openai(roi)

        results.append({
            "Question_cell(row,col)": (qr, qc),
            "Question_text": text
        })

    # ---------- SORT ----------
    results.sort(
        key=lambda x: (
            x["Question_cell(row,col)"][0],
            x["Question_cell(row,col)"][1]
        )
    )

    # ---------- SAVE ----------
    out_csv = folder / "OCR_questions_FINAL.csv"
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["Question_cell(row,col)", "Question_text"]
        )
        writer.writeheader()
        writer.writerows(results)

    print(f" Saved → {out_csv}")

print("\n OCR COMPLETED USING OPENAI VISION (FIXED)")


In [ ]:
csv_path = Path(
    "DS project/results_grid/input/OCR_questions_FINAL.csv"
)

print("Exists:", csv_path.exists())

df = pd.read_csv(csv_path)
df["Question_cell(row,col)"] = df["Question_cell(row,col)"].apply(ast.literal_eval)

df


In [ ]:
def clean_ocr_text(text: str) -> str:
    if not isinstance(text, str):
        return ""


    text = text.replace("\n", " ")

    text = re.sub(r"-\s+", "", text)


    text = re.sub(r"\s{2,}", " ", text)


    replacements = {
        "|": "",
        "Stré": "Strö",
        "Uber": "Über",
        "uber": "über",
        "ae": "ä",
        "oe": "ö",
        "ue": "ü",
        "Ae": "Ä",
        "Oe": "Ö",
        "Ue": "Ü",
    }

    for k, v in replacements.items():
        text = text.replace(k, v)

    return text.strip()



df = pd.read_csv(csv_path)


df["Question_cell(row,col)"] = df["Question_cell(row,col)"].apply(ast.literal_eval)


df["Question_text_clean"] = df["Question_text"].apply(clean_ocr_text)


df.to_csv(csv_path, index=False)

print(" Question_text_clean saved into the original CSV file")


In [ ]:
df = pd.read_csv(csv_path)

df["Question_cell(row,col)"] = df["Question_cell(row,col)"].apply(ast.literal_eval)


df["Question_text_clean"] = df["Question_text"].apply(clean_ocr_text)

df

In [ ]:
print(csv_path)

In [ ]:
# ===============================
# CONFIG
# ===============================
BASE_DIR = Path("DS project/results_grid")

OCR_CSV = BASE_DIR / "input" / "OCR_questions_FINAL.csv"
QA_CSV  = BASE_DIR / "input" / "FINAL_QA_COMPLETE.csv"

OUT_CSV = BASE_DIR / "input" / "final_file.csv"

# ===============================
# OCR CLEANING FUNCTION
# ===============================
def clean_ocr_text(text: str) -> str:
    if not isinstance(text, str):
        return ""


    text = text.replace("\n", " ")


    text = re.sub(r"-\s+", "", text)


    text = re.sub(r"\s{2,}", " ", text)


    replacements = {
        "|": "",
        "Uber": "Über",
        "uber": "über",
        "Stré": "Strö",
    }

    for k, v in replacements.items():
        text = text.replace(k, v)

    return text.strip()


# ===============================
# LOAD OCR QUESTIONS
# ===============================
df_ocr = pd.read_csv(OCR_CSV)
df_ocr["Question_cell(row,col)"] = df_ocr["Question_cell(row,col)"].apply(ast.literal_eval)
df_ocr["Question_text"] = df_ocr["Question_text"].apply(clean_ocr_text)

# ===============================
# LOAD QA METADATA
# ===============================
df_qa = pd.read_csv(QA_CSV)
df_qa["Question_cell(row,col)"] = df_qa["Question_cell(row,col)"].apply(ast.literal_eval)
df_qa["Answer_start_cell(row,col)"] = df_qa["Answer_start_cell(row,col)"].apply(ast.literal_eval)

# ===============================
# MERGE FILES
# ===============================
df_merged = pd.merge(
    df_qa,
    df_ocr[["Question_cell(row,col)", "Question_text"]],
    on="Question_cell(row,col)",
    how="left"
)

# ===============================
# SELECT & ORDER COLUMNS
# ===============================
df_merged = df_merged[
    [
        "Question_cell(row,col)",
        "Question_text",
        "Answer_start_cell(row,col)",
        "Direction",
        "Answer_length",
    ]
]

# ===============================
# SORT BY QUESTION COORDINATES
# ===============================
df_merged = df_merged.sort_values(
    by="Question_cell(row,col)",
    key=lambda x: x.apply(lambda t: (t[0], t[1]))
).reset_index(drop=True)

# ===============================
# SAVE RESULT
# ===============================
df_merged.to_csv(OUT_CSV, index=False, encoding="utf-8")

print(" DONE")
print(" Final merged file saved to:")
print(OUT_CSV)

# ===============================
# DISPLAY TABLE
# ===============================
df_merged


In [ ]:
# =========================
# CONFIG
# =========================

INPUT_FILE = "DS project/results_grid/input/final_file.csv"
OUTPUT_FILE = "DS project/results_grid/input/final_file_with_candidates.csv"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (CrosswordDebug/1.0)"
}

BLACKLIST = {"WÖRTERBUCH", "WORTLISTEN"}
DELAY = 0.5

# =========================
# Helpers
# =========================

def build_wort_suchen_url(clue: str, length: int):
    pattern = "_" * length
    clue_encoded = quote(clue.lower()).replace("%20", "+")
    return (
        "https://www.wort-suchen.de/"
        f"kreuzwortraetsel-hilfe/loesungen/{clue_encoded}/{pattern}/"
    )

def fetch_solutions(clue: str, length: int):
    url = build_wort_suchen_url(clue, length)
    r = requests.get(url, headers=HEADERS, timeout=20)
    soup = BeautifulSoup(r.text, "html.parser")

    sols = []
    for a in soup.find_all("a"):
        w = a.get_text(strip=True).upper()
        if w.isalpha() and len(w) == length and w not in BLACKLIST:
            sols.append(w)

    return list(dict.fromkeys(sols))

# =========================
# MAIN
# =========================

df = pd.read_csv(INPUT_FILE)

all_candidates = []

print("🔎 Fetching candidates and saving file...\n")

for idx, row in df.iterrows():
    clue = str(row["Question_text"]).strip()
    length = int(row["Answer_length"])

    cands = fetch_solutions(clue, length)
    all_candidates.append(cands)

    print(f"Q{idx}: {clue}")
    print("Candidates:", cands)
    print("-" * 40)

    time.sleep(DELAY)

df["candidates"] = all_candidates
df.to_csv(OUTPUT_FILE, index=False)

print("\n Saved:", OUTPUT_FILE)


In [ ]:
# =========================
# CONFIG
# =========================

INPUT_FILE = "DS project/results_grid/input/final_file_with_candidates.csv"
OUTPUT_FILE = "DS project/results_grid/input/final_file_solved_final.csv"

# =========================
# Helpers
# =========================

def parse_cell(cell_str):
    return tuple(map(int, cell_str.strip("()").split(",")))

def build_cells(start, direction, length):
    r, c = start
    cells = []
    for i in range(length):
        if direction.lower() == "right":
            cells.append((r, c + i))
        else:
            cells.append((r + i, c))
    return cells

def count_fixed_letters(q, grid):
    return sum(1 for cell in q["cells"] if cell in grid)

def fill_from_grid(q, grid):
    letters = []
    for cell in q["cells"]:
        letters.append(grid.get(cell, "_"))
    return "".join(letters)

# =========================
# Dynamic CSP Solver
# =========================

def solve_dynamic(questions, grid=None):
    if grid is None:
        grid = {}


    unsolved = [
        q for q in questions
        if q["answer"] is None and q["candidates"]
    ]


    if not unsolved:
        remaining = [q for q in questions if q["answer"] is None]
        if remaining:
            print("\n Remaining unanswered questions (no candidates):")
            for q in remaining:
                print(f"Q{q['id']} -> {q['clue']}")
        return True, grid


    q = max(
        unsolved,
        key=lambda x: count_fixed_letters(x, grid)
    )

    print(f"\n Choosing Q{q['id']} ({q['clue']})")
    print(f"Fixed letters: {count_fixed_letters(q, grid)}")
    print(f"Candidates: {q['candidates']}")

    for word in q["candidates"]:
        print(f"    Trying: {word}")

        conflict = False
        new_grid = dict(grid)

        for cell, ch in zip(q["cells"], word):
            if cell in new_grid and new_grid[cell] != ch:
                print(
                    f"       Conflict at {cell}: "
                    f"{new_grid[cell]} ≠ {ch}"
                )
                conflict = True
                break
            new_grid[cell] = ch

        if conflict:
            continue

        print(f"       Accepted: {word}")
        q["answer"] = word

        ok, result_grid = solve_dynamic(questions, new_grid)
        if ok:
            return True, result_grid

        # backtrack
        print(f"      Backtracking from {word}")
        q["answer"] = None

    print(f" No valid answer for Q{q['id']}")
    return False, grid

# =========================
# LOAD DATA
# =========================

df = pd.read_csv(INPUT_FILE)

questions = []

for idx, row in df.iterrows():
    start = parse_cell(row["Answer_start_cell(row,col)"])
    direction = row["Direction"]
    length = int(row["Answer_length"])

    q = {
        "id": idx,
        "clue": row["Question_text"],
        "cells": build_cells(start, direction, length),
        "candidates": ast.literal_eval(row["candidates"]),
        "answer": None,
    }
    questions.append(q)

# =========================
# RUN SOLVER
# =========================

print("\n Running dynamic crossword solver...\n")
ok, final_grid = solve_dynamic(questions)

print("\nSolved (partial allowed):", ok)
print("\n================ FINAL ANSWERS ================\n")

answers = []

for q in questions:
    if q["answer"] is not None:
        ans = q["answer"]
    else:
        ans = fill_from_grid(q, final_grid)
    answers.append(ans)
    print(f"Q{q['id']} → {ans}")

df["answers"] = answers

df.to_csv(OUTPUT_FILE, index=False)

print("\n Saved to:", OUTPUT_FILE)


In [ ]:
# ============================================================
# HYBRID CROSSWORD SOLVER (Wort-Suchen + LLM + CSP + Propagation) — FULL FIXED
#  Uses wort-suchen + LLM to harvest candidates
#  Never-empty rule: no question ends with 0 candidates (revive if empty)
#  Hard letter propagation loop (singleton answers force letters on crossings)
#  CSP backtracking AFTER grid is "warm" (has forced letters)
#  Saves final CSV with candidates + answers
# ============================================================


# =========================
# CONFIG
# =========================
INPUT_FILE = "DS project/results_grid/input/final_file.csv"
OUTPUT_FILE = "DS project/results_grid/input/final_file_solved_hybrid.csv"

HEADERS = {"User-Agent": "Mozilla/5.0 (CrosswordHybrid/1.0)"}
BLACKLIST = {"WÖRTERBUCH", "WORTLISTEN"}

DELAY_REF = 0.35
DELAY_LLM = 0.0

MAX_REF_CANDS = 30
MAX_LLM_CANDS = 30
MAX_TOTAL_CANDS = 40

TOPK_TRY_PER_VAR = 15  # BT tries per variable

# Weights
W_REF = 10
W_LLM = 4
W_REVIVE = 1
W_RECON = 2  # if you later want to store recon weight

# Propagation / recovery
PROP_MAX_ITERS = 40            # hard-propagation iterations cap
MIN_FIXED_FOR_RECON = 1        # if pattern has >= 1 fixed letter -> allow recon
RECON_WANT_N = 1               # ask LLM for ONE word in recon

# =========================
# HELPERS
# =========================

def parse_cell(cell_str: str) -> Tuple[int, int]:
    return tuple(map(int, cell_str.strip("()").split(",")))

def build_cells(start: Tuple[int, int], direction: str, length: int) -> List[Tuple[int, int]]:
    r, c = start
    cells = []
    if str(direction).lower() == "right":
        for i in range(length):
            cells.append((r, c + i))
    else:
        for i in range(length):
            cells.append((r + i, c))
    return cells

def normalize_clue(text: str) -> str:
    t = (text or "").strip()
    t = re.sub(r"\s+", " ", t)
    return t

def clamp_candidates(cands: List[str], length: int) -> List[str]:
    out = []
    seen = set()
    for c in cands or []:
        if not isinstance(c, str):
            continue
        w = c.strip().upper()
        if not w.isalpha():
            continue
        if len(w) != length:
            continue
        if w not in seen:
            seen.add(w)
            out.append(w)
    return out

def count_fixed_letters(q: Dict, grid: Dict[Tuple[int, int], str]) -> int:
    return sum(1 for cell in q["cells"] if cell in grid)

def fill_from_grid(q: Dict, grid: Dict[Tuple[int, int], str]) -> str:
    return "".join(grid.get(cell, "_") for cell in q["cells"])

# =========================
# WORT-SUCHEN
# =========================

def build_wort_suchen_url(clue: str, length: int) -> str:
    pattern = "_" * length
    clue_encoded = quote(clue.lower()).replace("%20", "+")
    return (
        "https://www.wort-suchen.de/"
        f"kreuzwortraetsel-hilfe/loesungen/{clue_encoded}/{pattern}/"
    )

def fetch_solutions(clue: str, length: int) -> List[str]:
    try:
        url = build_wort_suchen_url(clue, length)
        r = requests.get(url, headers=HEADERS, timeout=20)
        soup = BeautifulSoup(r.text, "html.parser")
    except Exception:
        return []

    sols = []
    for a in soup.find_all("a"):
        w = a.get_text(strip=True).upper()
        if w.isalpha() and len(w) == length and w not in BLACKLIST:
            sols.append(w)

    out = []
    seen = set()
    for w in sols:
        if w not in seen:
            seen.add(w)
            out.append(w)
    return out[:MAX_REF_CANDS]

# =========================
# OPENAI CLIENT
# =========================

OPENAI_MODEL = "gpt-4o"
TEMPERATURE = 0.2
MAX_OUT_TOKENS = 400

try:
    from openai import OpenAI
    openai_client = OpenAI()
except Exception:
    openai_client = None

def call_openai_text(prompt: str) -> str:
    if openai_client is None:
        raise RuntimeError("OpenAI client not initialized. Run: pip install -U openai")

    # New Responses API
    if hasattr(openai_client, "responses"):
        resp = openai_client.responses.create(
            model=OPENAI_MODEL,
            input=prompt,
            temperature=TEMPERATURE,
            max_output_tokens=MAX_OUT_TOKENS,
        )
        return (resp.output_text or "").strip()

    # Older chat completions
    if hasattr(openai_client, "chat") and hasattr(openai_client.chat, "completions"):
        resp = openai_client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=TEMPERATURE,
            max_tokens=MAX_OUT_TOKENS,
        )
        return resp.choices[0].message.content.strip()

    raise RuntimeError("Your openai package seems incompatible. Run: pip install -U openai")

# =========================
# LLM ASK
# =========================

def ask_llm(question_text: str, length: int, pattern: str, want_n: int = 30) -> List[str]:
    prompt = f"""
You are a professional German crossword puzzle constructor
with deep knowledge of STANDARD German crossword answers.

==================================================
CRITICAL MINDSET (READ CAREFULLY):

You are NOT generating free-form language.
You are generating answers that appear in REAL German crosswords.

If a clue has a well-known, specific crossword answer,
you MUST prefer it over generic descriptions.

Examples:
- "Fitnessübung, Dauerlauf" → TRIMMTRAB (NOT LAUFSTUND, NOT JOGGING)
- "sizilianischer Vulkan" → AETNA (NOT ETNAE, NOT ETNAS)
==================================================

CONTEXT (OCR WARNING):
The clue text comes from OCR and may contain mistakes:
- wrong letters (e.g., l/I, O/0, rn/m, ü/u, ä/a)
- missing or extra spaces
- broken words or line breaks
You must infer the intended clue meaning and silently correct OCR errors.

--------------------------------------------------
STRICT FORMAL RULES (ABSOLUTE):

- Answer must be ONE German word
- NO spaces, NO hyphens
- EXACTLY {length} letters
- Pattern letters MUST match exactly

PATTERN:
{pattern}

--------------------------------------------------
CROSSWORD NORMALIZATION RULES:

- Do NOT use Ä Ö Ü ß
- Write instead:
  Ä → AE | Ö → OE | Ü → UE | ß → SS
- The added E COUNTS as a full letter
- If the normalized spelling does NOT fit the length:
  DO NOT return the word

--------------------------------------------------
PROPER NOUNS & CROSSWORD SPELLING (VERY IMPORTANT):

Many famous names have a STANDARD crossword spelling.

Examples:
- ETNA → AETNA
- IRAQ → IRAK
- ROM → ROMA (if length requires)
- ATHENS → ATHEN

Rules:
- If the clue refers to a place, mountain, volcano, island, city, river:
  → Prefer the STANDARD crossword form
- NEVER invent suffixes
- NEVER generate random letter variants
- If only ONE standard answer exists, return ONLY that one

--------------------------------------------------
MEANING FIRST, LETTERS SECOND (CRITICAL):

Before generating answers:
1. Determine the clue type:
   - Definition (noun / verb / adjective)
   - Proper noun (place, volcano, city, etc.)
   - Abbreviation
   - Onomatopoeia
   - Archaic / crossword-style term

2. Ask yourself:
   - Is there a KNOWN crossword term for this?
   - Would a human crossword solver expect a SPECIFIC word?
   ARCHAIC CROSSWORD RULE (VERY IMPORTANT):

If the clue explicitly contains words like:
- "veraltet"
- "alt"
- "altertümlich"

Then:
- Actively search for OLD-FASHIONED German words.
- Prefer SHORT (3–5 letters) crossword stock words.
- These words may NOT appear in modern dictionaries.
- Such answers are VERY COMMON in German crosswords.

Examples:
- veraltet: gerade, eben → JUST
- veraltet: alldieweil → DERO
- veraltet: alldieweil, alldieweil → EHED


--------------------------------------------------
CANDIDATE GENERATION RULES:

- Generate ONLY semantically correct answers
- Every candidate must:
  - Make sense WITHOUT crossings
  - Be something a human would accept in a crossword
- Avoid vague or descriptive compounds
- Avoid artificial or invented forms
- NEVER generate answers by mutating letters

--------------------------------------------------
CANDIDATE STRATEGY:

- Generate MULTIPLE plausible candidates (if they exist)
- Order them from:
  MOST standard / MOST common
  → less common but still valid
- If ONLY ONE valid crossword answer exists:
  → return ONLY that answer
--------------------------------------------------
CONTROLLED DIVERSITY (VERY IMPORTANT):

Even if there is ONE very strong standard answer,
you SHOULD still generate additional valid alternatives
IF they are commonly accepted in German crosswords.

Guidelines:
- The FIRST candidate must always be the BEST / MOST STANDARD answer.
- After that, include SECONDARY but still VALID crossword answers:
  - synonyms
  - archaic forms
  - crossword-typical alternatives
- Do NOT invent words.
- Do NOT include vague or descriptive phrases.
- Every candidate must be something that could realistically
  appear in a printed crossword.

Quantity rules:
- If the clue is a definition (noun / verb / adjective):
  → try to return 5–10 candidates if possible
- If the clue is a proper noun:
  → return 1–3 candidates (only real known variants)
- If the clue is an abbreviation or onomatopoeia:
  → return all common forms used in crosswords

The goal is:
HIGH QUALITY first, then CONTROLLED DIVERSITY.
--------------------------------------------------
SEMANTIC INTERPRETATION STEP (CRITICAL):

First, identify the REAL-WORLD CONCEPT behind the clue
(e.g. organ, place, function, object).

Then generate answers that NAME this concept,
not only crossword-style abstractions.

If a concrete, common noun names the concept directly,
it MUST be included even if it is obvious.

AMBIGUOUS CLUE HANDLING (VERY IMPORTANT):

Some clues are VERY GENERAL and cannot be uniquely solved
without crossing letters.

Examples:
- "eine Blume"

For such clues:
- DO NOT try to guess a single "best" answer.
- Generate a BROAD, DIVERSE set of COMMON crossword answers.
- Prefer SHORT, SIMPLE, BASIC nouns.
- Assume the final choice will be determined by crossings.
CONCRETE OBJECT PRIORITY (CRITICAL):

If a clue describes a FUNCTION, ROLE, or PROPERTY of something,
you MUST consider the COMMON CONCRETE OBJECT that fulfills it.

Examples:
- "Sinnesorgan" → AUGE

These concrete nouns MUST be included
even if more abstract or technical terms also exist.

--------------------------------------------------
OUTPUT FORMAT (STRICT):

- Output ONLY a valid JSON array
- UPPERCASE words
- Exactly {want_n} or fewer entries
- NO explanations
- NO markdown
- NO comments

--------------------------------------------------
CLUE (OCR):
"{question_text}"

Return ONLY the JSON array.
"""
    text = call_openai_text(prompt)
    text = re.sub(r"```json|```", "", text).strip()
    try:
        arr = json.loads(text)
        return arr if isinstance(arr, list) else []
    except Exception:
        return []

# =========================
# CANDIDATE HARVEST
# =========================

def init_question_fields(q: Dict) -> None:
    # Defensive init so we never KeyError
    if "candidates" not in q or q["candidates"] is None:
        q["candidates"] = []
    if "weights" not in q or q["weights"] is None:
        q["weights"] = {}
    if "sources" not in q or q["sources"] is None:
        q["sources"] = {}
    if "answer" not in q:
        q["answer"] = None

def harvest_candidates_for_question(q: Dict) -> None:
    init_question_fields(q)

    clue = normalize_clue(q["clue"])
    length = q["length"]

    q["candidates"] = []
    q["weights"] = {}
    q["sources"] = {}

    # 1) Reference (wort-suchen)
    ref = clamp_candidates(fetch_solutions(clue, length), length)
    for w in ref:
        if w not in q["weights"]:
            q["candidates"].append(w)
            q["weights"][w] = W_REF
            q["sources"][w] = "ref"

    time.sleep(DELAY_REF)

    # 2) LLM
    llm = []
    try:
        llm = ask_llm(clue, length, "_" * length, want_n=MAX_LLM_CANDS)
    except Exception:
        llm = []

    llm = clamp_candidates(llm, length)
    for w in llm:
        if w not in q["weights"]:
            q["candidates"].append(w)
            q["weights"][w] = W_LLM
            q["sources"][w] = "llm"

    if DELAY_LLM:
        time.sleep(DELAY_LLM)

    q["candidates"] = q["candidates"][:MAX_TOTAL_CANDS]

def revive_candidates_if_empty(q: Dict, pattern: str) -> None:
    init_question_fields(q)

    if q["candidates"]:
        return

    clue = normalize_clue(q["clue"])
    length = q["length"]

    prompts = [
        clue,
        clue + " (standard crossword answer)",
        clue + " (more alternatives, real crossword stock, no inventions)",
    ]

    got = []
    for p in prompts:
        try:
            got.extend(ask_llm(p, length, pattern, want_n=60))
        except Exception:
            continue

    got = clamp_candidates(got, length)

    # absolute fallback (to avoid crash)
    if not got:
        got = ["X" * length]

    for w in got:
        if w not in q["weights"]:
            q["candidates"].append(w)
            q["weights"][w] = W_REVIVE
            q["sources"][w] = "revive"

    q["candidates"] = q["candidates"][:MAX_TOTAL_CANDS]

def ensure_nonempty_candidates(questions: List[Dict]) -> None:
    for q in questions:
        init_question_fields(q)
        if not q["candidates"]:
            revive_candidates_if_empty(q, "_" * q["length"])

# =========================
# GRID CONSISTENCY
# =========================

def word_fits_grid(q: Dict, word: str, grid: Dict[Tuple[int, int], str]) -> bool:
    for cell, ch in zip(q["cells"], word):
        if cell in grid and grid[cell] != ch:
            return False
    return True

def place_word_on_grid(q: Dict, word: str, grid: Dict[Tuple[int, int], str]) -> Optional[Dict[Tuple[int, int], str]]:
    new_grid = dict(grid)
    for cell, ch in zip(q["cells"], word):
        if cell in new_grid and new_grid[cell] != ch:
            return None
        new_grid[cell] = ch
    return new_grid

def build_grid_from_answers(questions: List[Dict]) -> Optional[Dict[Tuple[int, int], str]]:
    grid = {}
    for q in questions:
        if not q.get("answer"):
            continue
        for cell, ch in zip(q["cells"], q["answer"]):
            if cell in grid and grid[cell] != ch:
                return None
            grid[cell] = ch
    return grid

# =========================
# HARD PROPAGATION (THE “COLLECTING” IDEA)
# =========================

def hard_propagate_and_prune(questions: List[Dict]) -> bool:
    """
    Loop:
      - Build forced-letter grid from solved answers
      - Prune candidates that conflict
      - If a question becomes singleton => fix it as answer
    Returns False on contradiction.
    """
    ensure_nonempty_candidates(questions)

    for _ in range(PROP_MAX_ITERS):
        changed = False
        grid = build_grid_from_answers(questions)
        if grid is None:
            return False

        for q in questions:
            if q.get("answer") is not None:
                continue

            # prune by current forced letters
            keep = [w for w in q["candidates"] if word_fits_grid(q, w, grid)]

            if not keep:
                # revive using pattern from grid (best effort)
                patt = fill_from_grid(q, grid)
                revive_candidates_if_empty(q, patt)
                keep = [w for w in q["candidates"] if word_fits_grid(q, w, grid)]
                if not keep:
                    return False

            if len(keep) < len(q["candidates"]):
                q["candidates"] = keep
                changed = True

            if q.get("answer") is None and len(q["candidates"]) == 1:
                q["answer"] = q["candidates"][0]
                changed = True

        if not changed:
            return True

    return True

# =========================
# ORDERING HEURISTICS
# =========================

def count_intersections(q: Dict, questions: List[Dict]) -> int:
    s = set(q["cells"])
    return sum(1 for other in questions if other is not q and (s & set(other["cells"])))

def order_candidates_by_weight_and_fit(q: Dict, grid: Dict, topk: int) -> List[str]:
    init_question_fields(q)
    cands = [w for w in q["candidates"] if word_fits_grid(q, w, grid)]
    cands.sort(key=lambda w: q.get("weights", {}).get(w, 0), reverse=True)
    return cands[:topk]

# =========================
# SEED
# =========================

def seed_initial_answer(questions: List[Dict]) -> bool:
    """
    Choose a good anchor:
    - Prefer fewer candidates
    - Prefer longer words
    """
    ranked = sorted(
        questions,
        key=lambda q: (
            len(q.get("candidates", [])) if q.get("candidates") else 10**9,
            -q.get("length", 0),
        )
    )
    for q in ranked:
        if q.get("answer") is not None:
            continue
        if not q.get("candidates"):
            continue
        q["answer"] = q["candidates"][0]
        return True
    return False

# =========================
# CSP SOLVER (BACKTRACK)
# =========================

def solve_dynamic_hybrid(questions: List[Dict], grid: Optional[Dict] = None) -> Tuple[bool, Dict]:
    if grid is None:
        grid = {}

    # choose unsolved that still have candidates
    unsolved = [q for q in questions if q.get("answer") is None and q.get("candidates")]

    if not unsolved:
        return True, grid

    def var_key(x: Dict):
        fixed = count_fixed_letters(x, grid)
        feasible = [w for w in x["candidates"] if word_fits_grid(x, w, grid)]
        cand_count = len(feasible)
        inters = count_intersections(x, questions)
        # prefer: more fixed letters, fewer feasible options, more intersections
        return (fixed, -cand_count, inters)

    q = max(unsolved, key=var_key)

    ordered = order_candidates_by_weight_and_fit(q, grid, TOPK_TRY_PER_VAR)
    if not ordered:
        return False, grid

    for word in ordered:
        new_grid = place_word_on_grid(q, word, grid)
        if new_grid is None:
            continue

        q["answer"] = word

        # propagate after placing (key improvement)
        if not hard_propagate_and_prune(questions):
            q["answer"] = None
            continue

        ok, result_grid = solve_dynamic_hybrid(questions, new_grid)
        if ok:
            return True, result_grid

        # backtrack
        q["answer"] = None

        # undo side effects (simple reset: we re-propagate later anyway)
        # best effort: just continue. (If you want full rollback, implement snapshots.)

    return False, grid

# =========================
# FINAL GRID-BASED RECONSTRUCTION
# =========================

def reconstruct_word_from_pattern_llm(clue: str, length: int, pattern: str) -> Optional[str]:
    prompt = f"""
You are solving a German crossword.

CLUE:
{normalize_clue(clue)}

KNOWN LETTER PATTERN:
{pattern}

RULES:
- Return ONE German word only
- EXACTLY {length} letters
- Uppercase
- Must match the pattern
- No explanations

Return ONLY the word.
"""
    try:
        ans = call_openai_text(prompt).strip().upper()
    except Exception:
        return None

    if not (ans.isalpha() and len(ans) == length):
        return None

    for pch, ach in zip(pattern, ans):
        if pch != "_" and pch != ach:
            return None

    return ans

def final_recovery_pass(questions: List[Dict], grid: Dict[Tuple[int, int], str]) -> Dict[Tuple[int, int], str]:
    for q in questions:
        if q.get("answer") is not None:
            continue

        patt = fill_from_grid(q, grid)
        fixed_letters = q["length"] - patt.count("_")
        if fixed_letters < MIN_FIXED_FOR_RECON:
            continue

        w = reconstruct_word_from_pattern_llm(q["clue"], q["length"], patt)
        if w:
            q["answer"] = w
            for cell, ch in zip(q["cells"], w):
                grid[cell] = ch
    return grid

# =========================
# MAIN PIPELINE
# =========================

def run_hybrid_pipeline(input_file: str, output_file: str) -> None:
    df = pd.read_csv(input_file)

    # Build questions list
    questions: List[Dict] = []
    for idx, row in df.iterrows():
        start = parse_cell(row["Answer_start_cell(row,col)"])
        direction = str(row["Direction"])
        length = int(row["Answer_length"])
        clue = str(row["Question_text"]).strip()

        q = {
            "id": idx,
            "clue": clue,
            "cells": build_cells(start, direction, length),
            "length": length,
            "candidates": [],
            "weights": {},
            "sources": {},
            "answer": None,
        }
        questions.append(q)

    # 1) Harvest
    print("\n================ 1) HARVEST CANDIDATES (REF + LLM) ================\n")
    all_cands_for_df = []
    for q in questions:
        harvest_candidates_for_question(q)
        revive_candidates_if_empty(q, "_" * q["length"])
        all_cands_for_df.append(list(q["candidates"]))
        print(f"Q{q['id']} ({q['length']}): {q['clue']}")
        print(f"  cands: {q['candidates'][:12]}{'...' if len(q['candidates'])>12 else ''}")
        print("-" * 50)

    df["candidates"] = [str(c) for c in all_cands_for_df]

    # 2) Seed
    print("\n🌱 Seeding initial word...\n")
    seeded = seed_initial_answer(questions)
    if not seeded:
        print("⚠️ Could not seed any word. (No candidates).")
    else:
        for q in questions:
            if q.get("answer"):
                print(f"🌱 SEED Q{q['id']} = {q['answer']}")
                break

    # 3) Warm up grid with propagation BEFORE BT
    print("\n================ 2) PROPAGATE (HARD LETTERS) ================\n")
    ok_prop = hard_propagate_and_prune(questions)
    print("Propagation ok:", ok_prop)

    # Build initial grid from current answers
    grid0 = build_grid_from_answers(questions) or {}

    # 4) Solve (BT)
    print("\n================ 3) SOLVE (WEIGHTED CSP BACKTRACK) ================\n")
    ok, final_grid = solve_dynamic_hybrid(questions, grid=grid0)
    print("Solved (partial allowed):", ok)
    print("Fixed letters after BT:", len(final_grid))

    # 5) Recovery
    print("\n================ 4) FINAL RECOVERY (FROM GRID PATTERN) ================\n")
    final_grid = final_recovery_pass(questions, final_grid)
    print("Fixed letters after recovery:", len(final_grid))

    # 6) Outputs
    print("\n================ 5) FINAL ANSWERS ================\n")
    answers = []
    for q in questions:
        if q.get("answer") is not None:
            ans = q["answer"]
        else:
            ans = fill_from_grid(q, final_grid)
        answers.append(ans)
        print(f"Q{q['id']} → {ans}")

    df["answers"] = answers
    df.to_csv(output_file, index=False)
    print("\n✅ Saved:", output_file)

# =========================
# RUN
# =========================
if __name__ == "__main__":
    run_hybrid_pipeline(INPUT_FILE, OUTPUT_FILE)
